# Football Natural Language Question Answering Project

**Student:** Disath Tennakoon  
**Project:** Football NLP Question Answering  
**Notebook:** Shared NLP Pipeline  

## Project Objective

This project develops a hybrid Natural Language Question Answering system for international football data. The system uses an NLP classification model to identify the user's question intent and deterministic retrieval functions to obtain factual answers from structured football datasets.

This shared notebook implements:

- 7.1 Data Collection
- 7.2 Data Preprocessing
- 7.3 Exploratory Data Analysis
- Creation of fixed training, validation, and test datasets

After completing the shared stages, this notebook will be copied into two separate notebooks for:

1. Logistic Regression with TF-IDF
2. 1D Convolutional Neural Network with word embeddings

## Cell 2 — Mount Google Drive



In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Cell 3 — Imports and reproducibility

In [ ]:
import os
import random
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

print("Libraries imported successfully.")
print(f"Random seed: {RANDOM_SEED}")

Libraries imported successfully.
Random seed: 42


## Cell 4 — Define and verify file paths

In [ ]:
DATA_DIRECTORY = "/content/drive/MyDrive/NLP/data"

RESULTS_PATH = os.path.join(DATA_DIRECTORY, "results.csv")
GOALSCORERS_PATH = os.path.join(DATA_DIRECTORY, "goalscorers.csv")
SHOOTOUTS_PATH = os.path.join(DATA_DIRECTORY, "shootouts.csv")

dataset_paths = {
    "results": RESULTS_PATH,
    "goalscorers": GOALSCORERS_PATH,
    "shootouts": SHOOTOUTS_PATH
}

for dataset_name, file_path in dataset_paths.items():
    if os.path.exists(file_path):
        print(f"FOUND: {dataset_name} -> {file_path}")
    else:
        print(f"NOT FOUND: {dataset_name} -> {file_path}")

FOUND: results -> /content/drive/MyDrive/NLP/data/results.csv
FOUND: goalscorers -> /content/drive/MyDrive/NLP/data/goalscorers.csv
FOUND: shootouts -> /content/drive/MyDrive/NLP/data/shootouts.csv


## 7.1 Data Collection

### 7.1.1 Dataset Acquisition

Three structured football datasets are used as the factual knowledge source for this project:

- **results.csv** — International football match results and scores.
- **goalscorers.csv** — Goal-scoring events, including players and scoring minutes.
- **shootouts.csv** — Penalty-shootout results.

The datasets were stored in Google Drive and loaded into Google Colab for inspection and processing.

A separate manually created and labelled natural-language question dataset will be developed for intent classification. Its questions will be connected to the structured football records where appropriate.

## Cell 6 — Load the datasets

In [ ]:
results_df = pd.read_csv(RESULTS_PATH)
goalscorers_df = pd.read_csv(GOALSCORERS_PATH)
shootouts_df = pd.read_csv(SHOOTOUTS_PATH)

print("Datasets loaded successfully.")
print(f"Results dataset:     {results_df.shape[0]:,} rows × {results_df.shape[1]} columns")
print(f"Goalscorers dataset: {goalscorers_df.shape[0]:,} rows × {goalscorers_df.shape[1]} columns")
print(f"Shootouts dataset:   {shootouts_df.shape[0]:,} rows × {shootouts_df.shape[1]} columns")

Datasets loaded successfully.
Results dataset:     49,485 rows × 9 columns
Goalscorers dataset: 47,855 rows × 8 columns
Shootouts dataset:   682 rows × 5 columns


## Cell 7 — Display sample records

In [ ]:
display(results_df.head())
display(goalscorers_df.head())
display(shootouts_df.head())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
0,7/2/1916,Chile,Uruguay,Uruguay,José Piendibene,44,False,False
1,7/2/1916,Chile,Uruguay,Uruguay,Isabelino Gradín,55,False,False
2,7/2/1916,Chile,Uruguay,Uruguay,Isabelino Gradín,70,False,False
3,7/2/1916,Chile,Uruguay,Uruguay,José Piendibene,75,False,False
4,7/6/1916,Argentina,Chile,Argentina,Alberto Ohaco,2,False,False


,date,home_team,away_team,winner,first_shooter
0,8/22/1967,India,Taiwan,Taiwan,NaN
1,11/14/1971,South Korea,Vietnam Republic,South Korea,NaN
2,5/7/1972,South Korea,Iraq,Iraq,NaN
3,5/17/1972,Thailand,South Korea,South Korea,NaN
4,5/19/1972,Thailand,Cambodia,Thailand,NaN


## Cell 8 — Inspect structure and data types

In [ ]:
datasets = {
    "Results": results_df,
    "Goalscorers": goalscorers_df,
    "Shootouts": shootouts_df
}

for dataset_name, dataframe in datasets.items():
    print(f"\n{'=' * 60}")
    print(f"{dataset_name.upper()} DATASET")
    print(f"{'=' * 60}")

    dataframe.info()

    print("\nColumn data types:")
    display(
        dataframe.dtypes
        .rename("data_type")
        .reset_index()
        .rename(columns={"index": "column"})
    )


RESULTS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49485 entries, 0 to 49484
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   date        49485 non-null  object
 1   home_team   49485 non-null  object
 2   away_team   49485 non-null  object
 3   home_score  49485 non-null  int64 
 4   away_score  49485 non-null  int64 
 5   tournament  49485 non-null  object
 6   city        49485 non-null  object
 7   country     49485 non-null  object
 8   neutral     49485 non-null  bool  
dtypes: bool(1), int64(2), object(6)
memory usage: 3.1+ MB

Column data types:


,column,data_type
0,date,object
1,home_team,object
2,away_team,object
3,home_score,int64
4,away_score,int64
5,tournament,object
6,city,object
7,country,object
8,neutral,bool



GOALSCORERS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47855 entries, 0 to 47854
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   date       47855 non-null  object
 1   home_team  47855 non-null  object
 2   away_team  47855 non-null  object
 3   team       47855 non-null  object
 4   scorer     47807 non-null  object
 5   minute     47599 non-null  object
 6   own_goal   47855 non-null  bool  
 7   penalty    47855 non-null  bool  
dtypes: bool(2), object(6)
memory usage: 2.3+ MB

Column data types:


,column,data_type
0,date,object
1,home_team,object
2,away_team,object
3,team,object
4,scorer,object
5,minute,object
6,own_goal,bool
7,penalty,bool



SHOOTOUTS DATASET
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 682 entries, 0 to 681
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   date           682 non-null    object
 1   home_team      682 non-null    object
 2   away_team      682 non-null    object
 3   winner         682 non-null    object
 4   first_shooter  260 non-null    object
dtypes: object(5)
memory usage: 26.8+ KB

Column data types:


,column,data_type
0,date,object
1,home_team,object
2,away_team,object
3,winner,object
4,first_shooter,object


### 7.1.2 Dataset Description

The project uses three structured football datasets as its factual knowledge base.

#### Results dataset

The `results.csv` dataset contains 49,485 international football matches with nine attributes. Each row represents one match and includes the match date, home and away teams, final scores, tournament, location, and whether the match was played at a neutral venue.

#### Goalscorers dataset

The `goalscorers.csv` dataset contains 47,855 goal-event records with eight attributes. Each row represents a recorded goal and includes the match date, participating teams, scoring team, scorer, scoring minute, and whether the goal was an own goal or penalty.

The number of goal-event rows is not expected to equal the number of matches because a match can contain multiple goals or no goals.

#### Shootouts dataset

The `shootouts.csv` dataset contains 682 penalty-shootout records with five attributes. Each row represents one recorded shootout and includes the match date, participating teams, shootout winner, and the team that took the first kick where this information is available.

Together, these datasets support factual questions about match winners, match scores, goalscorers, scoring minutes, penalty-shootout winners, and player goal totals.

## Cell 10 — Create a column-description table

In [ ]:
column_descriptions = pd.DataFrame([
    ["results", "date", "Date on which the match was played"],
    ["results", "home_team", "Team listed as the home team"],
    ["results", "away_team", "Team listed as the away team"],
    ["results", "home_score", "Final score of the home team"],
    ["results", "away_score", "Final score of the away team"],
    ["results", "tournament", "Competition or match category"],
    ["results", "city", "City where the match was played"],
    ["results", "country", "Country where the match was played"],
    ["results", "neutral", "Whether the match was played at a neutral venue"],

    ["goalscorers", "date", "Date of the match"],
    ["goalscorers", "home_team", "Team listed as the home team"],
    ["goalscorers", "away_team", "Team listed as the away team"],
    ["goalscorers", "team", "Team credited with the goal"],
    ["goalscorers", "scorer", "Name of the recorded goalscorer"],
    ["goalscorers", "minute", "Recorded scoring minute, possibly including stoppage time"],
    ["goalscorers", "own_goal", "Whether the event was recorded as an own goal"],
    ["goalscorers", "penalty", "Whether the goal was scored from a penalty"],

    ["shootouts", "date", "Date of the match"],
    ["shootouts", "home_team", "Team listed as the home team"],
    ["shootouts", "away_team", "Team listed as the away team"],
    ["shootouts", "winner", "Winner of the penalty shootout"],
    ["shootouts", "first_shooter", "Team that took the first kick, when recorded"]
], columns=["dataset", "column", "description"])

display(column_descriptions)

,dataset,column,description
0,results,date,Date on which the match was played
1,results,home_team,Team listed as the home team
2,results,away_team,Team listed as the away team
3,results,home_score,Final score of the home team
4,results,away_score,Final score of the away team
5,results,tournament,Competition or match category
6,results,city,City where the match was played
7,results,country,Country where the match was played
8,results,neutral,Whether the match was played at a neutral venue
9,goalscorers,date,Date of the match


## Cell 11 — Missing-value inspection

In [ ]:
missing_value_reports = []

for dataset_name, dataframe in datasets.items():
    missing_count = dataframe.isna().sum()
    missing_percentage = (missing_count / len(dataframe) * 100).round(2)

    report = pd.DataFrame({
        "dataset": dataset_name,
        "column": dataframe.columns,
        "missing_count": missing_count.values,
        "missing_percentage": missing_percentage.values
    })

    missing_value_reports.append(report)

missing_values_summary = pd.concat(
    missing_value_reports,
    ignore_index=True
)

display(missing_values_summary)

,dataset,column,missing_count,missing_percentage
0,Results,date,0,0.00
1,Results,home_team,0,0.00
2,Results,away_team,0,0.00
3,Results,home_score,0,0.00
4,Results,away_score,0,0.00
5,Results,tournament,0,0.00
6,Results,city,0,0.00
7,Results,country,0,0.00
8,Results,neutral,0,0.00
9,Goalscorers,date,0,0.00


## Cell 12 — Duplicate inspection

In [ ]:
duplicate_summary = []

for dataset_name, dataframe in datasets.items():
    exact_duplicates = dataframe.duplicated().sum()

    duplicate_summary.append({
        "dataset": dataset_name,
        "total_rows": len(dataframe),
        "exact_duplicate_rows": exact_duplicates,
        "duplicate_percentage": round(
            exact_duplicates / len(dataframe) * 100, 3
        )
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)
display(duplicate_summary_df)

,dataset,total_rows,exact_duplicate_rows,duplicate_percentage
0,Results,49485,0,0.000
1,Goalscorers,47855,82,0.171
2,Shootouts,682,0,0.000


## Cell 12A — Compact duplicate counts

In [ ]:
duplicate_summary_df = pd.DataFrame([
    {
        "dataset": dataset_name,
        "total_rows": len(dataframe),
        "exact_duplicate_rows": int(dataframe.duplicated().sum()),
        "rows_in_duplicate_groups": int(
            dataframe.duplicated(keep=False).sum()
        )
    }
    for dataset_name, dataframe in datasets.items()
])

display(duplicate_summary_df)

,dataset,total_rows,exact_duplicate_rows,rows_in_duplicate_groups
0,Results,49485,0,0
1,Goalscorers,47855,82,128
2,Shootouts,682,0,0


## Cell 13 — Inspect any exact duplicates

In [ ]:
for dataset_name, dataframe in datasets.items():
    duplicate_rows = dataframe[
        dataframe.duplicated(keep=False)
    ].sort_values(list(dataframe.columns))

    print(f"\n{dataset_name}: {len(duplicate_rows):,} rows involved in exact duplicates")

    if not duplicate_rows.empty:
        display(duplicate_rows.head(20))


Results: 0 rows involved in exact duplicates

Goalscorers: 128 rows involved in exact duplicates


,date,home_team,away_team,team,scorer,minute,own_goal,penalty
5105,1/16/1968,Congo,Ghana,Ghana,Osei Kofi,NaN,False,False
5106,1/16/1968,Congo,Ghana,Ghana,Osei Kofi,NaN,False,False
5124,1/19/1968,Ghana,Ivory Coast,Ghana,Wilberforce Mfum,NaN,False,False
5125,1/19/1968,Ghana,Ivory Coast,Ghana,Wilberforce Mfum,NaN,False,False
5128,1/19/1968,Ghana,Ivory Coast,Ivory Coast,Laurent Pokou,NaN,False,False
5129,1/19/1968,Ghana,Ivory Coast,Ivory Coast,Laurent Pokou,NaN,False,False
22148,1/30/2002,Burkina Faso,Ghana,Ghana,Isaac Boakye,90,False,False
22149,1/30/2002,Burkina Faso,Ghana,Ghana,Isaac Boakye,90,False,False
41775,10/12/2021,Syria,Lebanon,Lebanon,Mohamad Kdouh,45,False,False
41776,10/12/2021,Syria,Lebanon,Lebanon,Mohamad Kdouh,45,False,False



Shootouts: 0 rows involved in exact duplicates


## Cell 14 — Inspect raw date formats and coverage

In [ ]:
# Inspect representative raw date values from different parts of each dataset
for dataset_name, dataframe in datasets.items():
    print(f"\n{dataset_name} date samples:")
    display(
        pd.concat([
            dataframe["date"].head(3),
            dataframe["date"].iloc[len(dataframe) // 2:
                                   len(dataframe) // 2 + 3],
            dataframe["date"].tail(3)
        ]).drop_duplicates()
    )


Results date samples:


,date
0,1872-11-30
1,1873-03-08
2,1874-03-07
24742,7/16/2000
49482,7/15/2026
49483,7/18/2026
49484,7/19/2026



Goalscorers date samples:


,date
0,7/2/1916
23927,6/20/2004
47852,7/18/2026
47854,7/19/2026



Shootouts date samples:


,date
0,8/22/1967
1,11/14/1971
2,5/7/1972
341,7/21/2004
342,7/25/2004
343,7/30/2004
679,6/29/2026
680,7/3/2026
681,7/7/2026


## Cell 14A — Parse dates using mixed-format detection

In [ ]:
date_summary = []

for dataset_name, dataframe in datasets.items():
    parsed_dates = pd.to_datetime(
        dataframe["date"].astype(str).str.strip(),
        format="mixed",
        errors="coerce"
    )

    date_summary.append({
        "dataset": dataset_name,
        "earliest_date": parsed_dates.min(),
        "latest_date": parsed_dates.max(),
        "unparseable_dates": int(parsed_dates.isna().sum())
    })

date_summary_df = pd.DataFrame(date_summary)
display(date_summary_df)

,dataset,earliest_date,latest_date,unparseable_dates
0,Results,1872-11-30,2026-07-19,0
1,Goalscorers,1916-07-02,2026-07-19,0
2,Shootouts,1967-08-22,2026-07-07,0


## Cell 15 — Inspect unusual goal-minute formats

In [ ]:
minute_text = goalscorers_df["minute"].dropna().astype(str).str.strip()

non_standard_minutes = minute_text[
    ~minute_text.str.fullmatch(r"\d+")
]

print(f"Recorded minute values: {len(minute_text):,}")
print(f"Non-standard minute values: {len(non_standard_minutes):,}")
print("\nSample non-standard values:")

display(
    non_standard_minutes
    .value_counts()
    .rename_axis("minute_value")
    .reset_index(name="frequency")
    .head(30)
)

Recorded minute values: 47,599
Non-standard minute values: 16

Sample non-standard values:


,minute_value,frequency
0,90+2,3
1,90+1,2
2,90+8,2
3,90+5,1
4,120+5,1
5,90+4,1
6,90+10,1
7,90+3,1
8,45+2,1
9,120+1,1


### 7.1.3 Initial Data-Quality Findings

Initial inspection identified the following data-quality characteristics:

#### Date formats

The `results.csv` dataset uses mixed date representations. Earlier records use the ISO-style `YYYY-MM-DD` format, while later records use the `M/D/YYYY` format. The `goalscorers.csv` and `shootouts.csv` datasets use `M/D/YYYY`.

After mixed-format parsing:

- Results coverage: 30 November 1872 to 19 July 2026
- Goalscorer coverage: 2 July 1916 to 19 July 2026
- Shootout coverage: 22 August 1967 to 7 July 2026
- No date values were unparseable

The different starting dates indicate that goalscorer and shootout information is unavailable for many historical matches. Therefore, the absence of a linked record must not automatically be interpreted as proof that no goal or shootout occurred.

#### Missing values

The results dataset contains no missing values. However:

- 48 goalscorer records have no recorded scorer
- 256 goalscorer records have no recorded scoring minute
- 422 shootout records have no recorded first-shooter team

These records will be preserved. Missing factual values will be handled using an honest `data incomplete` response rather than imputation or invention.

#### Duplicate records

No exact duplicate rows were identified in the results or shootouts datasets. The goalscorers dataset contains 82 exact duplicate occurrences beyond the first record, with 128 rows belonging to duplicate groups.

These rows will not be removed automatically because two goals by the same player in the same match may appear identical when the scoring minute is unavailable. Their effect will be evaluated before any deduplication decision is made.

#### Scoring-minute formats

Most scoring-minute values contain a single integer. Sixteen records use valid stoppage-time notation, including values such as `45+2`, `90+8`, and `120+5`. The original notation will be preserved during preprocessing.

## Cell 17 — Create temporary standardized match keys

In [ ]:
def parse_football_dates(date_series):
    """
    Parse ISO YYYY-MM-DD and US-style M/D/YYYY dates explicitly.
    This avoids relying on ambiguous automatic date inference.
    """
    date_text = date_series.astype(str).str.strip()

    iso_mask = date_text.str.fullmatch(r"\d{4}-\d{1,2}-\d{1,2}")

    parsed_dates = pd.Series(
        pd.NaT,
        index=date_series.index,
        dtype="datetime64[ns]"
    )

    parsed_dates.loc[iso_mask] = pd.to_datetime(
        date_text.loc[iso_mask],
        format="%Y-%m-%d",
        errors="coerce"
    )

    parsed_dates.loc[~iso_mask] = pd.to_datetime(
        date_text.loc[~iso_mask],
        format="%m/%d/%Y",
        errors="coerce"
    )

    return parsed_dates


results_link_df = results_df.copy()
goalscorers_link_df = goalscorers_df.copy()
shootouts_link_df = shootouts_df.copy()

for dataframe in [
    results_link_df,
    goalscorers_link_df,
    shootouts_link_df
]:
    dataframe["date_parsed"] = parse_football_dates(dataframe["date"])

    dataframe["home_team_key"] = (
        dataframe["home_team"]
        .astype(str)
        .str.strip()
        .str.casefold()
    )

    dataframe["away_team_key"] = (
        dataframe["away_team"]
        .astype(str)
        .str.strip()
        .str.casefold()
    )

MATCH_KEY = [
    "date_parsed",
    "home_team_key",
    "away_team_key"
]

print("Temporary standardized linkage fields created.")

Temporary standardized linkage fields created.


## Cell 18 — Confirm unique match keys in results

In [ ]:
results_key_duplicates = results_link_df.duplicated(
    subset=MATCH_KEY,
    keep=False
)

print(
    "Results rows sharing the same date/home/away key:",
    f"{results_key_duplicates.sum():,}"
)

if results_key_duplicates.any():
    display(
        results_link_df.loc[
            results_key_duplicates,
            [
                "date",
                "home_team",
                "away_team",
                "home_score",
                "away_score",
                "tournament"
            ]
        ]
        .sort_values(["date", "home_team", "away_team"])
        .head(30)
    )

Results rows sharing the same date/home/away key: 4


,date,home_team,away_team,home_score,away_score,tournament
9641,2/17/1974,Tahiti,New Caledonia,2,1,Friendly
9642,2/17/1974,Tahiti,New Caledonia,1,2,Friendly
49352,6/6/2026,Gibraltar,Cayman Islands,4,1,Friendly
49361,6/6/2026,Gibraltar,Cayman Islands,4,1,Friendly


## Cell 19 — Test goalscorer-to-results linkage

In [ ]:
result_match_keys = (
    results_link_df[MATCH_KEY]
    .drop_duplicates()
    .assign(match_found=True)
)

goalscorer_linkage = goalscorers_link_df.merge(
    result_match_keys,
    on=MATCH_KEY,
    how="left"
)

goalscorer_linkage_summary = pd.DataFrame([{
    "dataset": "Goalscorers",
    "total_records": len(goalscorer_linkage),
    "linked_records": int(goalscorer_linkage["match_found"].notna().sum()),
    "unlinked_records": int(goalscorer_linkage["match_found"].isna().sum()),
    "linkage_rate_percentage": round(
        goalscorer_linkage["match_found"].notna().mean() * 100,
        2
    )
}])

display(goalscorer_linkage_summary)

,dataset,total_records,linked_records,unlinked_records,linkage_rate_percentage
0,Goalscorers,47855,47839,16,99.97


## Cell 20 — Test shootout-to-results linkage

In [ ]:
shootout_linkage = shootouts_link_df.merge(
    result_match_keys,
    on=MATCH_KEY,
    how="left"
)

shootout_linkage_summary = pd.DataFrame([{
    "dataset": "Shootouts",
    "total_records": len(shootout_linkage),
    "linked_records": int(shootout_linkage["match_found"].notna().sum()),
    "unlinked_records": int(shootout_linkage["match_found"].isna().sum()),
    "linkage_rate_percentage": round(
        shootout_linkage["match_found"].notna().mean() * 100,
        2
    )
}])

display(shootout_linkage_summary)

,dataset,total_records,linked_records,unlinked_records,linkage_rate_percentage
0,Shootouts,682,681,1,99.85


## Cell 21 — Inspect unlinked records

In [ ]:
unlinked_goalscorers = goalscorer_linkage[
    goalscorer_linkage["match_found"].isna()
]

unlinked_shootouts = shootout_linkage[
    shootout_linkage["match_found"].isna()
]

print(
    f"Unlinked goalscorer records: {len(unlinked_goalscorers):,}"
)
if not unlinked_goalscorers.empty:
    display(
        unlinked_goalscorers[
            ["date", "home_team", "away_team", "team", "scorer"]
        ].head(20)
    )

print(
    f"\nUnlinked shootout records: {len(unlinked_shootouts):,}"
)
if not unlinked_shootouts.empty:
    display(
        unlinked_shootouts[
            ["date", "home_team", "away_team", "winner"]
        ].head(20)
    )

Unlinked goalscorer records: 16


,date,home_team,away_team,team,scorer
47639,6/20/2026,Netherlands,Sweden,Netherlands,Cody Gakpo
47640,6/20/2026,Netherlands,Sweden,Netherlands,Brian Brobbey
47641,6/20/2026,Netherlands,Sweden,Netherlands,Brian Brobbey
47642,6/20/2026,Netherlands,Sweden,Netherlands,Brian Brobbey
47643,6/20/2026,Netherlands,Sweden,Netherlands,Tijjani Reijnders
47644,6/20/2026,Netherlands,Sweden,Sweden,Alexander Isak
47645,6/20/2026,Tunisia,Japan,Japan,Ayase Ueda
47646,6/20/2026,Tunisia,Japan,Japan,Ayase Ueda
47647,6/20/2026,Tunisia,Japan,Japan,Daichi Kamada
47648,6/20/2026,Tunisia,Japan,Japan,Daichi Kamada



Unlinked shootout records: 1


,date,home_team,away_team,winner
422,6/29/2011,Saare County,Åland Islands,Åland Islands


### 7.1.4 Cross-Dataset Linkage Findings

The datasets were linked using a standardized composite match key containing:

- Parsed match date
- Normalized home-team name
- Normalized away-team name

The linkage analysis produced the following results:

- 47,839 of 47,855 goalscorer records linked to the results dataset, giving a linkage rate of 99.97%.
- 681 of 682 shootout records linked successfully, giving a linkage rate of 99.85%.
- Sixteen goalscorer records could not be linked. These records represent four matches dated 20 and 25 June 2026.
- One shootout record, involving Saare County and Åland Islands on 29 June 2011, could not be linked.

Four results rows share a date/home-team/away-team key. These rows represent two ambiguous match-key groups:

1. Tahiti versus New Caledonia on 17 February 1974, where two different scores are recorded.
2. Gibraltar versus Cayman Islands on 6 June 2026, where the same score appears in two separate result records.

Consequently, a date and two team names do not uniquely identify every match in the results dataset. Ambiguous match groups and unlinked records will be excluded from automatic question generation unless they can be resolved using additional attributes. This prevents contradictory or unsupported answers from entering the labelled question dataset.

## Cell 23 — Inspect complete ambiguous result records

In [ ]:
ambiguous_results = (
    results_link_df.loc[results_key_duplicates]
    .sort_values(MATCH_KEY)
)

display(
    ambiguous_results[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "tournament",
            "city",
            "country",
            "neutral"
        ]
    ]
)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
9641,2/17/1974,Tahiti,New Caledonia,2,1,Friendly,Papeete,Tahiti,False
9642,2/17/1974,Tahiti,New Caledonia,1,2,Friendly,Papeete,Tahiti,False
49352,6/6/2026,Gibraltar,Cayman Islands,4,1,Friendly,Gibraltar,Gibraltar,False
49361,6/6/2026,Gibraltar,Cayman Islands,4,1,Friendly,Europa Point,Gibraltar,False


## Cell 24 — Create safe and ambiguous result keys

In [ ]:
result_key_counts = (
    results_link_df
    .groupby(MATCH_KEY, dropna=False)
    .size()
    .reset_index(name="result_key_count")
)

safe_result_keys = result_key_counts[
    result_key_counts["result_key_count"] == 1
][MATCH_KEY].copy()

ambiguous_result_keys = result_key_counts[
    result_key_counts["result_key_count"] > 1
][MATCH_KEY].copy()

safe_results_df = results_link_df.merge(
    safe_result_keys,
    on=MATCH_KEY,
    how="inner"
).copy()

print(f"Unique result records available: {len(safe_results_df):,}")
print(f"Ambiguous match-key groups: {len(ambiguous_result_keys):,}")
print(
    "Result rows excluded because of ambiguous keys:",
    f"{len(results_link_df) - len(safe_results_df):,}"
)

Unique result records available: 49,481
Ambiguous match-key groups: 2
Result rows excluded because of ambiguous keys: 4


## Cell 25 — Create safely linked event datasets

In [ ]:
safe_goalscorers_df = goalscorers_link_df.merge(
    safe_result_keys.assign(safe_result_key=True),
    on=MATCH_KEY,
    how="inner"
).copy()

safe_shootouts_df = shootouts_link_df.merge(
    safe_result_keys.assign(safe_result_key=True),
    on=MATCH_KEY,
    how="inner"
).copy()

print(
    "Safely linked goalscorer records:",
    f"{len(safe_goalscorers_df):,}"
)
print(
    "Goalscorer records excluded:",
    f"{len(goalscorers_link_df) - len(safe_goalscorers_df):,}"
)

print(
    "Safely linked shootout records:",
    f"{len(safe_shootouts_df):,}"
)
print(
    "Shootout records excluded:",
    f"{len(shootout_linkage) - len(safe_shootouts_df):,}"
)

Safely linked goalscorer records: 47,839
Goalscorer records excluded: 16
Safely linked shootout records: 681
Shootout records excluded: 1


## Cell 26 — Analyze exact goalscorer duplicates

In [ ]:
goalscorer_duplicate_groups = (
    goalscorers_link_df[
        goalscorers_link_df.duplicated(
            subset=list(goalscorers_df.columns),
            keep=False
        )
    ]
    .groupby(
        list(goalscorers_df.columns),
        dropna=False
    )
    .size()
    .reset_index(name="occurrence_count")
    .sort_values(
        ["occurrence_count", "date"],
        ascending=[False, True]
    )
)

print(
    "Exact duplicate groups:",
    f"{len(goalscorer_duplicate_groups):,}"
)

display(
    goalscorer_duplicate_groups[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "minute",
            "own_goal",
            "penalty",
            "occurrence_count"
        ]
    ].head(30)
)

Exact duplicate groups: 46


,date,home_team,away_team,team,scorer,minute,own_goal,penalty,occurrence_count
27,2/27/1980,Solomon Islands,Tahiti,Tahiti,NaN,NaN,False,False,12
28,2/28/1980,New Caledonia,Papua New Guinea,New Caledonia,NaN,NaN,False,False,8
30,2/29/1980,Fiji,Tahiti,Tahiti,NaN,NaN,False,False,6
9,11/25/1968,Haiti,Trinidad and Tobago,Trinidad and Tobago,Warren Archibald,NaN,False,False,4
20,2/24/1980,Vanuatu,Papua New Guinea,Papua New Guinea,NaN,NaN,False,False,4
24,2/26/1980,Australia,Papua New Guinea,Australia,Peter Sharne,NaN,False,False,4
25,2/26/1980,New Caledonia,Vanuatu,New Caledonia,NaN,NaN,False,False,4
17,2/24/1980,New Caledonia,Australia,Australia,Eddie Krncevic,NaN,False,False,3
21,2/24/1980,Vanuatu,Papua New Guinea,Vanuatu,NaN,NaN,False,False,3
22,2/25/1980,Fiji,Solomon Islands,Fiji,NaN,NaN,False,False,3


## Cell 27 — Summarize duplicate groups by minute availability

In [ ]:
duplicate_minute_summary = pd.DataFrame([
    {
        "duplicate_group_type": "Minute recorded",
        "number_of_groups": int(
            goalscorer_duplicate_groups["minute"].notna().sum()
        ),
        "total_rows": int(
            goalscorer_duplicate_groups.loc[
                goalscorer_duplicate_groups["minute"].notna(),
                "occurrence_count"
            ].sum()
        )
    },
    {
        "duplicate_group_type": "Minute missing",
        "number_of_groups": int(
            goalscorer_duplicate_groups["minute"].isna().sum()
        ),
        "total_rows": int(
            goalscorer_duplicate_groups.loc[
                goalscorer_duplicate_groups["minute"].isna(),
                "occurrence_count"
            ].sum()
        )
    }
])

display(duplicate_minute_summary)

,duplicate_group_type,number_of_groups,total_rows
0,Minute recorded,10,20
1,Minute missing,36,108


### 7.1.5 Safe Record Selection

Four result records were excluded from automatic question generation because their date, home team, and away team did not form unique match identifiers.

The Tahiti versus New Caledonia records from 17 February 1974 contain contradictory scores despite sharing the same tournament and location. The two Gibraltar versus Cayman Islands records from 6 June 2026 have the same score but different city values. Although the latter records may represent separately recorded fixtures, a natural-language question containing only the date and teams would not distinguish between them reliably.

After applying the safe match-key rule:

- 49,481 unique result records remained available
- 47,839 goalscorer records remained safely linked
- 681 penalty-shootout records remained safely linked
- 16 unlinked goalscorer records were excluded
- One unlinked shootout record was excluded

This filtering prioritizes answer reliability over retaining every available record.

## Cell 29 — Compare goal-event counts with final scores

In [ ]:
original_goalscorer_columns = list(goalscorers_df.columns)

safe_results_score_df = safe_results_df[
    MATCH_KEY + ["home_score", "away_score"]
].copy()

safe_results_score_df["result_total_goals"] = (
    safe_results_score_df["home_score"]
    + safe_results_score_df["away_score"]
)

raw_goal_counts = (
    safe_goalscorers_df
    .groupby(MATCH_KEY, dropna=False)
    .size()
    .reset_index(name="raw_goal_event_count")
)

goal_count_comparison = safe_results_score_df.merge(
    raw_goal_counts,
    on=MATCH_KEY,
    how="left"
)

goal_count_comparison["raw_goal_event_count"] = (
    goal_count_comparison["raw_goal_event_count"]
    .fillna(0)
    .astype(int)
)

goal_count_comparison["raw_count_matches_score"] = (
    goal_count_comparison["raw_goal_event_count"]
    == goal_count_comparison["result_total_goals"]
)

display(
    goal_count_comparison[
        ["result_total_goals", "raw_goal_event_count",
         "raw_count_matches_score"]
    ]
    .agg({
        "result_total_goals": "count",
        "raw_goal_event_count": "sum",
        "raw_count_matches_score": "sum"
    })
    .to_frame(name="value")
)

,value
result_total_goals,49481
raw_goal_event_count,47839
raw_count_matches_score,19485


## Cell 30 — Create a conservative duplicate-cleaning candidate

In [ ]:
minute_recorded_mask = safe_goalscorers_df["minute"].notna()

recorded_minute_rows = safe_goalscorers_df[
    minute_recorded_mask
].drop_duplicates(
    subset=original_goalscorer_columns,
    keep="first"
)

missing_minute_rows = safe_goalscorers_df[
    ~minute_recorded_mask
]

goalscorers_candidate_df = pd.concat(
    [recorded_minute_rows, missing_minute_rows],
    ignore_index=True
)

print(
    "Safely linked goalscorer rows before candidate cleaning:",
    f"{len(safe_goalscorers_df):,}"
)
print(
    "Rows after removing recorded-minute exact duplicates:",
    f"{len(goalscorers_candidate_df):,}"
)
print(
    "Candidate duplicate rows removed:",
    f"{len(safe_goalscorers_df) - len(goalscorers_candidate_df):,}"
)

Safely linked goalscorer rows before candidate cleaning: 47,839
Rows after removing recorded-minute exact duplicates: 47,829
Candidate duplicate rows removed: 10


## Cell 31 — Test whether candidate cleaning improves score agreement

In [ ]:
candidate_goal_counts = (
    goalscorers_candidate_df
    .groupby(MATCH_KEY, dropna=False)
    .size()
    .reset_index(name="candidate_goal_event_count")
)

goal_count_comparison = goal_count_comparison.merge(
    candidate_goal_counts,
    on=MATCH_KEY,
    how="left"
)

goal_count_comparison["candidate_goal_event_count"] = (
    goal_count_comparison["candidate_goal_event_count"]
    .fillna(0)
    .astype(int)
)

goal_count_comparison["candidate_count_matches_score"] = (
    goal_count_comparison["candidate_goal_event_count"]
    == goal_count_comparison["result_total_goals"]
)

comparison_summary = pd.DataFrame([
    {
        "version": "Original goalscorer records",
        "matching_matches": int(
            goal_count_comparison["raw_count_matches_score"].sum()
        ),
        "nonmatching_matches": int(
            (~goal_count_comparison["raw_count_matches_score"]).sum()
        ),
        "match_percentage": round(
            goal_count_comparison["raw_count_matches_score"].mean() * 100,
            2
        )
    },
    {
        "version": "Recorded-minute duplicates removed",
        "matching_matches": int(
            goal_count_comparison[
                "candidate_count_matches_score"
            ].sum()
        ),
        "nonmatching_matches": int(
            (~goal_count_comparison[
                "candidate_count_matches_score"
            ]).sum()
        ),
        "match_percentage": round(
            goal_count_comparison[
                "candidate_count_matches_score"
            ].mean() * 100,
            2
        )
    }
])

display(comparison_summary)

,version,matching_matches,nonmatching_matches,match_percentage
0,Original goalscorer records,19485,29996,39.38
1,Recorded-minute duplicates removed,19475,30006,39.36


## Cell 32 — Inspect affected matches

In [ ]:
affected_goal_counts = goal_count_comparison[
    goal_count_comparison["raw_goal_event_count"]
    != goal_count_comparison["candidate_goal_event_count"]
].copy()

affected_goal_counts["raw_difference_from_score"] = (
    affected_goal_counts["raw_goal_event_count"]
    - affected_goal_counts["result_total_goals"]
)

affected_goal_counts["candidate_difference_from_score"] = (
    affected_goal_counts["candidate_goal_event_count"]
    - affected_goal_counts["result_total_goals"]
)

# Add only descriptive match fields.
# Scores are already present in affected_goal_counts.
match_descriptions = safe_results_df[
    MATCH_KEY + [
        "date",
        "home_team",
        "away_team"
    ]
].copy()

affected_goal_counts = affected_goal_counts.merge(
    match_descriptions,
    on=MATCH_KEY,
    how="left"
)

display(
    affected_goal_counts[
        [
            "date",
            "home_team",
            "away_team",
            "home_score",
            "away_score",
            "raw_goal_event_count",
            "candidate_goal_event_count",
            "raw_difference_from_score",
            "candidate_difference_from_score"
        ]
    ].sort_values("date")
)

,date,home_team,away_team,home_score,away_score,raw_goal_event_count,candidate_goal_event_count,raw_difference_from_score,candidate_difference_from_score
2,1/30/2002,Burkina Faso,Ghana,1,2,3,2,0,-1
8,10/12/2021,Syria,Lebanon,2,3,5,4,0,-1
4,10/9/2004,Turkey,Kazakhstan,4,0,4,3,0,-1
3,12/3/2003,Maldives,Mongolia,12,0,12,11,0,-1
1,4/30/2001,Oman,Laos,12,0,12,11,0,-1
5,6/13/2015,Poland,Georgia,4,0,4,3,0,-1
0,6/6/1981,Fiji,Taiwan,2,1,3,2,0,-1
9,9/25/2022,Moldova,Liechtenstein,2,0,2,1,0,-1
7,9/5/2021,San Marino,Poland,1,7,8,7,0,-1
6,9/8/2019,Spain,Faroe Islands,4,0,4,3,0,-1


## Cell 33 — Inspect missing-minute duplicate groups against match scores

In [ ]:
missing_minute_duplicate_groups = (
    goalscorer_duplicate_groups[
        goalscorer_duplicate_groups["minute"].isna()
    ]
    .copy()
)

missing_minute_duplicate_groups["date_parsed"] = (
    parse_football_dates(
        missing_minute_duplicate_groups["date"]
    )
)

missing_minute_duplicate_groups["home_team_key"] = (
    missing_minute_duplicate_groups["home_team"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

missing_minute_duplicate_groups["away_team_key"] = (
    missing_minute_duplicate_groups["away_team"]
    .astype(str)
    .str.strip()
    .str.casefold()
)

missing_duplicate_score_check = (
    missing_minute_duplicate_groups.merge(
        safe_results_df[
            MATCH_KEY + ["home_score", "away_score"]
        ],
        on=MATCH_KEY,
        how="left"
    )
)

missing_duplicate_score_check["result_total_goals"] = (
    missing_duplicate_score_check["home_score"]
    + missing_duplicate_score_check["away_score"]
)

display(
    missing_duplicate_score_check[
        [
            "date",
            "home_team",
            "away_team",
            "team",
            "scorer",
            "occurrence_count",
            "home_score",
            "away_score",
            "result_total_goals"
        ]
    ]
    .sort_values(
        ["occurrence_count", "date"],
        ascending=[False, True]
    )
    .head(40)
)

,date,home_team,away_team,team,scorer,occurrence_count,home_score,away_score,result_total_goals
0,2/27/1980,Solomon Islands,Tahiti,Tahiti,NaN,12,1,12,13
1,2/28/1980,New Caledonia,Papua New Guinea,New Caledonia,NaN,8,8,0,8
2,2/29/1980,Fiji,Tahiti,Tahiti,NaN,6,3,6,9
3,11/25/1968,Haiti,Trinidad and Tobago,Trinidad and Tobago,Warren Archibald,4,2,4,6
4,2/24/1980,Vanuatu,Papua New Guinea,Papua New Guinea,NaN,4,3,4,7
5,2/26/1980,Australia,Papua New Guinea,Australia,Peter Sharne,4,11,2,13
6,2/26/1980,New Caledonia,Vanuatu,New Caledonia,NaN,4,4,3,7
7,2/24/1980,New Caledonia,Australia,Australia,Eddie Krncevic,3,0,8,8
8,2/24/1980,Vanuatu,Papua New Guinea,Vanuatu,NaN,3,3,4,7
9,2/25/1980,Fiji,Solomon Islands,Fiji,NaN,3,3,1,4


### 7.1.6 Final Goalscorer Duplicate-Handling Policy

The goalscorer dataset contains 128 rows belonging to 46 exact duplicate groups. Of these groups:

- 10 groups contain a recorded scoring minute, representing 20 rows
- 36 groups have no recorded scoring minute, representing 108 rows

An experimental cleaning operation removed the repeated occurrence from each recorded-minute duplicate group. This reduced the goalscorer dataset by ten rows. However, the number of matches whose goal-event count agreed with the recorded final score decreased from 19,485 to 19,475.

All ten affected matches originally had goal-event counts equal to their final total scores. After removing the repeated records, each match became short by exactly one goal. Therefore, the repeated recorded-minute rows represent genuine separate scoring events rather than accidental duplicate data.

The missing-minute duplicate groups also show meaningful goal multiplicity. In several matches, repeated records correspond to plausible multi-goal performances or collectively account for the match's recorded total score.

Consequently:

- No goalscorer rows will be removed solely because they are exact duplicates.
- Each goalscorer row will be treated as a separate goal event.
- Repeated rows will retain their multiplicity during answer generation.
- Missing scorer or minute values will remain missing rather than being imputed.
- Only the 16 records that cannot be linked safely to a result will be excluded.

This evidence-based policy prevents genuine goals from being incorrectly deleted.

## Cell 35 — Select the final safe working datasets

In [ ]:
# Preserve every safely linked goal event, including repeated rows.
final_results_df = safe_results_df.copy()
final_goalscorers_df = safe_goalscorers_df.copy()
final_shootouts_df = safe_shootouts_df.copy()

print("Final safe dataset sizes:")
print(f"Results: {len(final_results_df):,}")
print(f"Goalscorers: {len(final_goalscorers_df):,}")
print(f"Shootouts: {len(final_shootouts_df):,}")

assert len(final_results_df) == 49_481
assert len(final_goalscorers_df) == 47_839
assert len(final_shootouts_df) == 681

print("\nFinal safe dataset counts validated.")

Final safe dataset sizes:
Results: 49,481
Goalscorers: 47,839
Shootouts: 681

Final safe dataset counts validated.


## Cell 36 — Create final standardized date and team fields

In [ ]:
for dataframe in [
    final_results_df,
    final_goalscorers_df,
    final_shootouts_df
]:
    dataframe["date_standardized"] = (
        dataframe["date_parsed"]
        .dt.strftime("%Y-%m-%d")
    )

    dataframe["home_team_standardized"] = (
        dataframe["home_team"]
        .astype(str)
        .str.strip()
    )

    dataframe["away_team_standardized"] = (
        dataframe["away_team"]
        .astype(str)
        .str.strip()
    )

print("Final standardized date and team fields created.")

Final standardized date and team fields created.


## Cell 37 — Process scoring-minute values safely

In [ ]:
def parse_scoring_minute(value):
    """
    Convert a football-minute value into separate components without
    discarding its original notation.
    """
    if pd.isna(value):
        return pd.Series({
            "minute_original": pd.NA,
            "base_minute": pd.NA,
            "added_time": pd.NA,
            "minute_total": pd.NA
        })

    minute_text = str(value).strip()

    if "+" in minute_text:
        base_text, added_text = minute_text.split("+", maxsplit=1)

        try:
            base_minute = int(float(base_text))
            added_time = int(float(added_text))
        except ValueError:
            base_minute = pd.NA
            added_time = pd.NA
            minute_total = pd.NA
        else:
            minute_total = base_minute + added_time

    else:
        try:
            base_minute = int(float(minute_text))
        except ValueError:
            base_minute = pd.NA
            minute_total = pd.NA
        else:
            minute_total = base_minute

        added_time = 0 if pd.notna(base_minute) else pd.NA

    return pd.Series({
        "minute_original": minute_text,
        "base_minute": base_minute,
        "added_time": added_time,
        "minute_total": minute_total
    })


minute_fields = final_goalscorers_df["minute"].apply(
    parse_scoring_minute
)

final_goalscorers_df = pd.concat(
    [
        final_goalscorers_df.reset_index(drop=True),
        minute_fields.reset_index(drop=True)
    ],
    axis=1
)

for column in ["base_minute", "added_time", "minute_total"]:
    final_goalscorers_df[column] = (
        final_goalscorers_df[column].astype("Int64")
    )

print("Scoring-minute fields created.")
display(
    final_goalscorers_df[
        [
            "minute",
            "minute_original",
            "base_minute",
            "added_time",
            "minute_total"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["minute_total", "minute_original"],
        na_position="last"
    )
    .tail(20)
)

Scoring-minute fields created.


,minute,minute_original,base_minute,added_time,minute_total
1086,105,105,105,0,105
7734,106,106,106,0,106
4132,107,107,107,0,107
5910,108,108,108,0,108
797,109,109,109,0,109
4133,110,110,110,0,110
1088,111,111,111,0,111
963,112,112,112,0,112
3301,113,113,113,0,113
1832,114,114,114,0,114


## 7.2 Supported Question Intents

Based on the available fields and data-quality findings, the first version of the football question-answering system will support the following intents.

### 1. Match winner

Example:

> Who won the match between Brazil and Germany on 2014-07-08?

Required data:

- Match date
- Home team
- Away team
- Home score
- Away score

Possible answers:

- Home team won
- Away team won
- The match ended in a draw
- Match not found
- Ambiguous match

### 2. Match score

Example:

> What was the score between Brazil and Germany on 2014-07-08?

Required data:

- Match date
- Home team
- Away team
- Home score
- Away score

### 3. Goal scorer

Example:

> Who scored in the match between Brazil and Germany on 2014-07-08?

Required data:

- Safely linked goalscorer records
- Recorded scorer names

If some scorer values are missing, the response must state that the available scorer information is incomplete.

### 4. Player goal count in a match

Example:

> How many goals did Miroslav Klose score against Brazil on 2014-07-08?

Each goalscorer row represents one separate goal event. Therefore, repeated identical rows must be counted rather than removed.

### 5. Goal minute

Example:

> When did Miroslav Klose score against Brazil on 2014-07-08?

The original football notation, including stoppage-time values such as `90+2`, will be used in the answer. If the minute is unavailable, the system will report that the scoring minute is not recorded.

### 6. Penalty shootout winner

Example:

> Who won the penalty shootout between Brazil and Chile on 2014-06-28?

Required data:

- Safely linked shootout record
- Recorded shootout winner

### 7. Tournament-based match retrieval

Example:

> Which team won the match between France and Croatia in the 2018 FIFA World Cup final?

This intent can use tournament information together with the team names and date or other available identifying information.

### Reliability boundaries

The system will not:

- Invent missing scorer or scoring-minute information
- Treat an absent historical goalscorer record as proof that no goal was scored
- Generate questions from ambiguous result keys
- Generate questions from unlinked goalscorer or shootout records
- Remove repeated goal events solely because their visible fields are identical
- Answer broad player-career totals unless the required dataset coverage is verified

### 7.2.1 Scoring-Minute Standardization

Scoring-minute values were processed without replacing the original data.

Four fields were retained or created:

- `minute`: the value from the original dataset
- `minute_original`: the original football-minute notation stored as text
- `base_minute`: the regulation or extra-time minute
- `added_time`: the recorded stoppage-time component
- `minute_total`: the numerical sum of the base minute and added time

For example, `120+1` is represented as:

- Base minute: 120
- Added time: 1
- Total minute: 121

Missing minute values remain missing. Unusual values are preserved because automatically correcting them without supporting evidence could introduce errors.

## Cell 40 — Validate records used for result-based questions

In [ ]:
result_qa_validation = pd.DataFrame([
    {
        "check": "Safe result records",
        "value": len(final_results_df)
    },
    {
        "check": "Duplicate safe match keys",
        "value": int(
            final_results_df.duplicated(
                subset=MATCH_KEY,
                keep=False
            ).sum()
        )
    },
    {
        "check": "Missing standardized dates",
        "value": int(
            final_results_df["date_standardized"].isna().sum()
        )
    },
    {
        "check": "Missing home scores",
        "value": int(
            final_results_df["home_score"].isna().sum()
        )
    },
    {
        "check": "Missing away scores",
        "value": int(
            final_results_df["away_score"].isna().sum()
        )
    },
    {
        "check": "Negative home scores",
        "value": int(
            (final_results_df["home_score"] < 0).sum()
        )
    },
    {
        "check": "Negative away scores",
        "value": int(
            (final_results_df["away_score"] < 0).sum()
        )
    }
])

display(result_qa_validation)

,check,value
0,Safe result records,49481
1,Duplicate safe match keys,0
2,Missing standardized dates,0
3,Missing home scores,0
4,Missing away scores,0
5,Negative home scores,0
6,Negative away scores,0


## Cell 41 — Prepare the result-question source table

In [ ]:
result_question_source = final_results_df[
    [
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "home_score",
        "away_score",
        "tournament",
        "city",
        "country",
        "neutral"
    ]
].copy()

result_question_source["home_score"] = (
    result_question_source["home_score"].astype(int)
)

result_question_source["away_score"] = (
    result_question_source["away_score"].astype(int)
)

result_question_source["match_id"] = [
    f"MATCH_{number:05d}"
    for number in range(1, len(result_question_source) + 1)
]

display(result_question_source.head())

,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,tournament,city,country,neutral,match_id
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,MATCH_00001
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,MATCH_00002
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,MATCH_00003
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,MATCH_00004
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,MATCH_00005


## Cell 42 — Generate match-winner questions

In [ ]:
def create_match_winner_answer(row):
    home_team = row["home_team_standardized"]
    away_team = row["away_team_standardized"]
    home_score = row["home_score"]
    away_score = row["away_score"]
    match_date = row["date_standardized"]

    if home_score > away_score:
        return (
            f"{home_team} won against {away_team} "
            f"{home_score}-{away_score} on {match_date}."
        )

    if away_score > home_score:
        return (
            f"{away_team} won against {home_team} "
            f"{away_score}-{home_score} on {match_date}."
        )

    return (
        f"The match between {home_team} and {away_team} "
        f"ended in a {home_score}-{away_score} draw "
        f"on {match_date}."
    )


winner_qa_df = result_question_source.copy()

winner_qa_df["question_id"] = [
    f"WIN_{number:05d}"
    for number in range(1, len(winner_qa_df) + 1)
]

winner_qa_df["intent"] = "match_winner"

winner_qa_df["question"] = (
    "Who won the match between "
    + winner_qa_df["home_team_standardized"]
    + " and "
    + winner_qa_df["away_team_standardized"]
    + " on "
    + winner_qa_df["date_standardized"]
    + "?"
)

winner_qa_df["answer"] = winner_qa_df.apply(
    create_match_winner_answer,
    axis=1
)

winner_qa_df = winner_qa_df[
    [
        "question_id",
        "match_id",
        "intent",
        "question",
        "answer",
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "home_score",
        "away_score",
        "tournament"
    ]
]

print(f"Match-winner questions generated: {len(winner_qa_df):,}")
display(winner_qa_df.head(10))

Match-winner questions generated: 49,481


,question_id,match_id,intent,question,answer,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,tournament
0,WIN_00001,MATCH_00001,match_winner,Who won the match between Scotland and England on 1872-11-30?,The match between Scotland and England ended in a 0-0 draw on 1872-11-30.,1872-11-30,Scotland,England,0,0,Friendly
1,WIN_00002,MATCH_00002,match_winner,Who won the match between England and Scotland on 1873-03-08?,England won against Scotland 4-2 on 1873-03-08.,1873-03-08,England,Scotland,4,2,Friendly
2,WIN_00003,MATCH_00003,match_winner,Who won the match between Scotland and England on 1874-03-07?,Scotland won against England 2-1 on 1874-03-07.,1874-03-07,Scotland,England,2,1,Friendly
3,WIN_00004,MATCH_00004,match_winner,Who won the match between England and Scotland on 1875-03-06?,The match between England and Scotland ended in a 2-2 draw on 1875-03-06.,1875-03-06,England,Scotland,2,2,Friendly
4,WIN_00005,MATCH_00005,match_winner,Who won the match between Scotland and England on 1876-03-04?,Scotland won against England 3-0 on 1876-03-04.,1876-03-04,Scotland,England,3,0,Friendly
5,WIN_00006,MATCH_00006,match_winner,Who won the match between Scotland and Wales on 1876-03-25?,Scotland won against Wales 4-0 on 1876-03-25.,1876-03-25,Scotland,Wales,4,0,Friendly
6,WIN_00007,MATCH_00007,match_winner,Who won the match between England and Scotland on 1877-03-03?,Scotland won against England 3-1 on 1877-03-03.,1877-03-03,England,Scotland,1,3,Friendly
7,WIN_00008,MATCH_00008,match_winner,Who won the match between Wales and Scotland on 1877-03-05?,Scotland won against Wales 2-0 on 1877-03-05.,1877-03-05,Wales,Scotland,0,2,Friendly
8,WIN_00009,MATCH_00009,match_winner,Who won the match between Scotland and England on 1878-03-02?,Scotland won against England 7-2 on 1878-03-02.,1878-03-02,Scotland,England,7,2,Friendly
9,WIN_00010,MATCH_00010,match_winner,Who won the match between Scotland and Wales on 1878-03-23?,Scotland won against Wales 9-0 on 1878-03-23.,1878-03-23,Scotland,Wales,9,0,Friendly


# Cell 43 — Generate match-score questions

In [ ]:
score_qa_df = result_question_source.copy()

score_qa_df["question_id"] = [
    f"SCORE_{number:05d}"
    for number in range(1, len(score_qa_df) + 1)
]

score_qa_df["intent"] = "match_score"

score_qa_df["question"] = (
    "What was the score between "
    + score_qa_df["home_team_standardized"]
    + " and "
    + score_qa_df["away_team_standardized"]
    + " on "
    + score_qa_df["date_standardized"]
    + "?"
)

score_qa_df["answer"] = (
    score_qa_df["home_team_standardized"]
    + " "
    + score_qa_df["home_score"].astype(str)
    + "-"
    + score_qa_df["away_score"].astype(str)
    + " "
    + score_qa_df["away_team_standardized"]
    + "."
)

score_qa_df = score_qa_df[
    [
        "question_id",
        "match_id",
        "intent",
        "question",
        "answer",
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "home_score",
        "away_score",
        "tournament"
    ]
]

print(f"Match-score questions generated: {len(score_qa_df):,}")
display(score_qa_df.head(10))

Match-score questions generated: 49,481


,question_id,match_id,intent,question,answer,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,tournament
0,SCORE_00001,MATCH_00001,match_score,What was the score between Scotland and England on 1872-11-30?,Scotland 0-0 England.,1872-11-30,Scotland,England,0,0,Friendly
1,SCORE_00002,MATCH_00002,match_score,What was the score between England and Scotland on 1873-03-08?,England 4-2 Scotland.,1873-03-08,England,Scotland,4,2,Friendly
2,SCORE_00003,MATCH_00003,match_score,What was the score between Scotland and England on 1874-03-07?,Scotland 2-1 England.,1874-03-07,Scotland,England,2,1,Friendly
3,SCORE_00004,MATCH_00004,match_score,What was the score between England and Scotland on 1875-03-06?,England 2-2 Scotland.,1875-03-06,England,Scotland,2,2,Friendly
4,SCORE_00005,MATCH_00005,match_score,What was the score between Scotland and England on 1876-03-04?,Scotland 3-0 England.,1876-03-04,Scotland,England,3,0,Friendly
5,SCORE_00006,MATCH_00006,match_score,What was the score between Scotland and Wales on 1876-03-25?,Scotland 4-0 Wales.,1876-03-25,Scotland,Wales,4,0,Friendly
6,SCORE_00007,MATCH_00007,match_score,What was the score between England and Scotland on 1877-03-03?,England 1-3 Scotland.,1877-03-03,England,Scotland,1,3,Friendly
7,SCORE_00008,MATCH_00008,match_score,What was the score between Wales and Scotland on 1877-03-05?,Wales 0-2 Scotland.,1877-03-05,Wales,Scotland,0,2,Friendly
8,SCORE_00009,MATCH_00009,match_score,What was the score between Scotland and England on 1878-03-02?,Scotland 7-2 England.,1878-03-02,Scotland,England,7,2,Friendly
9,SCORE_00010,MATCH_00010,match_score,What was the score between Scotland and Wales on 1878-03-23?,Scotland 9-0 Wales.,1878-03-23,Scotland,Wales,9,0,Friendly


## Cell 44 — Combine the first two intents

In [ ]:
result_based_qa_df = pd.concat(
    [
        winner_qa_df,
        score_qa_df
    ],
    ignore_index=True
)

intent_summary = (
    result_based_qa_df
    .groupby("intent")
    .size()
    .reset_index(name="question_count")
)

print(
    "Total result-based questions:",
    f"{len(result_based_qa_df):,}"
)

display(intent_summary)

Total result-based questions: 98,962


,intent,question_count
0,match_score,49481
1,match_winner,49481


## Cell 45 — Validate the generated questions and answers

In [ ]:
qa_validation_summary = pd.DataFrame([
    {
        "check": "Total questions",
        "value": len(result_based_qa_df)
    },
    {
        "check": "Duplicate question IDs",
        "value": int(
            result_based_qa_df["question_id"].duplicated().sum()
        )
    },
    {
        "check": "Duplicate question text",
        "value": int(
            result_based_qa_df["question"].duplicated().sum()
        )
    },
    {
        "check": "Missing questions",
        "value": int(
            result_based_qa_df["question"].isna().sum()
        )
    },
    {
        "check": "Empty questions",
        "value": int(
            result_based_qa_df["question"]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
    },
    {
        "check": "Missing answers",
        "value": int(
            result_based_qa_df["answer"].isna().sum()
        )
    },
    {
        "check": "Empty answers",
        "value": int(
            result_based_qa_df["answer"]
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
    }
])

display(qa_validation_summary)

assert len(result_based_qa_df) == 98_962
assert result_based_qa_df["question_id"].is_unique
assert result_based_qa_df["question"].is_unique
assert result_based_qa_df["question"].notna().all()
assert result_based_qa_df["answer"].notna().all()

print("Initial QA dataset validation passed.")

,check,value
0,Total questions,98962
1,Duplicate question IDs,0
2,Duplicate question text,0
3,Missing questions,0
4,Empty questions,0
5,Missing answers,0
6,Empty answers,0


Initial QA dataset validation passed.


## Cell 46 — Inspect random examples from both intents

In [ ]:
for intent_name in ["match_winner", "match_score"]:
    print(f"\nIntent: {intent_name}")

    display(
        result_based_qa_df[
            result_based_qa_df["intent"] == intent_name
        ][
            [
                "question_id",
                "question",
                "answer",
                "tournament"
            ]
        ].sample(
            n=5,
            random_state=42
        )
    )


Intent: match_winner


,question_id,question,answer,tournament
21035,WIN_21036,Who won the match between Jamaica and Saint Kitts and Nevis on 1996-05-26?,Jamaica won against Saint Kitts and Nevis 4-1 on 1996-05-26.,CFU Caribbean Cup
22618,WIN_22619,Who won the match between Cameroon and Algeria on 1998-02-15?,Cameroon won against Algeria 2-1 on 1998-02-15.,African Cup of Nations
40575,WIN_40576,Who won the match between Burkina Faso and Benin on 2017-05-04?,The match between Burkina Faso and Benin ended in a 1-1 draw on 2017-05-04.,Friendly
17983,WIN_17984,Who won the match between Mali and Tunisia on 1991-09-26?,The match between Mali and Tunisia ended in a 0-0 draw on 1991-09-26.,Friendly
35166,WIN_35167,Who won the match between Zimbabwe and Liberia on 2011-09-04?,Zimbabwe won against Liberia 3-0 on 2011-09-04.,African Cup of Nations qualification



Intent: match_score


,question_id,question,answer,tournament
70516,SCORE_21036,What was the score between Jamaica and Saint Kitts and Nevis on 1996-05-26?,Jamaica 4-1 Saint Kitts and Nevis.,CFU Caribbean Cup
72099,SCORE_22619,What was the score between Cameroon and Algeria on 1998-02-15?,Cameroon 2-1 Algeria.,African Cup of Nations
90056,SCORE_40576,What was the score between Burkina Faso and Benin on 2017-05-04?,Burkina Faso 1-1 Benin.,Friendly
67464,SCORE_17984,What was the score between Mali and Tunisia on 1991-09-26?,Mali 0-0 Tunisia.,Friendly
84647,SCORE_35167,What was the score between Zimbabwe and Liberia on 2011-09-04?,Zimbabwe 3-0 Liberia.,African Cup of Nations qualification


## 7.3 Initial Result-Based Question Generation

Two reliable question intents were generated from the final safe results dataset:

- `match_winner`: 49,481 questions
- `match_score`: 49,481 questions

This produced 98,962 question-answer pairs in total.

Each safe match was assigned a unique `match_id`. Questions were also assigned intent-specific identifiers using the prefixes `WIN_` and `SCORE_`.

Validation confirmed that the generated dataset contains:

- No duplicate question identifiers
- No duplicate question text
- No missing or empty questions
- No missing or empty answers
- Exactly two questions for every safe match

The initial wording uses a single controlled template for each intent. Additional linguistic variations will be introduced later without changing the factual answers.

During model-data splitting, all questions sharing the same `match_id` must remain in the same split. This prevents information about one match from appearing in both training and evaluation data.

## Cell 48 — Connect safe goalscorer records to match_id

In [ ]:
match_id_lookup = final_results_df[
    MATCH_KEY
].copy()

match_id_lookup["match_id"] = result_question_source["match_id"].values

goalscorer_question_source = final_goalscorers_df.merge(
    match_id_lookup,
    on=MATCH_KEY,
    how="left",
    validate="many_to_one"
)

print(
    "Goalscorer rows connected to match IDs:",
    f"{goalscorer_question_source['match_id'].notna().sum():,}"
)
print(
    "Goalscorer rows without match IDs:",
    f"{goalscorer_question_source['match_id'].isna().sum():,}"
)

assert goalscorer_question_source["match_id"].notna().all()
assert len(goalscorer_question_source) == 47_839

print("All final goalscorer records connected successfully.")

Goalscorer rows connected to match IDs: 47,839
Goalscorer rows without match IDs: 0
All final goalscorer records connected successfully.


## Cell 49 — Measure goalscorer-field completeness

In [ ]:
goalscorer_field_validation = pd.DataFrame([
    {
        "check": "Total safe goal events",
        "value": len(goalscorer_question_source)
    },
    {
        "check": "Missing scoring team",
        "value": int(
            goalscorer_question_source["team"].isna().sum()
        )
    },
    {
        "check": "Missing scorer name",
        "value": int(
            goalscorer_question_source["scorer"].isna().sum()
        )
    },
    {
        "check": "Missing scoring minute",
        "value": int(
            goalscorer_question_source["minute_original"].isna().sum()
        )
    },
    {
        "check": "Own-goal events",
        "value": int(
            goalscorer_question_source["own_goal"].fillna(False).sum()
        )
    },
    {
        "check": "Penalty-goal events",
        "value": int(
            goalscorer_question_source["penalty"].fillna(False).sum()
        )
    }
])

display(goalscorer_field_validation)

,check,value
0,Total safe goal events,47839
1,Missing scoring team,0
2,Missing scorer name,48
3,Missing scoring minute,256
4,Own-goal events,926
5,Penalty-goal events,3259


## Cell 50 — Measure goalscorer coverage by match

In [ ]:
goalscorer_match_coverage = (
    goalscorer_question_source
    .groupby("match_id")
    .agg(
        recorded_goal_events=("match_id", "size"),
        known_scorer_events=("scorer", lambda values: values.notna().sum()),
        missing_scorer_events=("scorer", lambda values: values.isna().sum()),
        known_minute_events=(
            "minute_original",
            lambda values: values.notna().sum()
        ),
        missing_minute_events=(
            "minute_original",
            lambda values: values.isna().sum()
        )
    )
    .reset_index()
)

goalscorer_match_coverage["all_scorers_known"] = (
    goalscorer_match_coverage["missing_scorer_events"] == 0
)

goalscorer_match_coverage["some_scorers_known"] = (
    goalscorer_match_coverage["known_scorer_events"] > 0
)

goalscorer_match_coverage["all_minutes_known"] = (
    goalscorer_match_coverage["missing_minute_events"] == 0
)

match_coverage_summary = pd.DataFrame([
    {
        "check": "Safe result matches",
        "value": len(final_results_df)
    },
    {
        "check": "Matches with at least one goalscorer record",
        "value": goalscorer_match_coverage["match_id"].nunique()
    },
    {
        "check": "Matches with at least one known scorer",
        "value": int(
            goalscorer_match_coverage["some_scorers_known"].sum()
        )
    },
    {
        "check": "Matches with all recorded scorers known",
        "value": int(
            goalscorer_match_coverage["all_scorers_known"].sum()
        )
    },
    {
        "check": "Matches containing missing scorer names",
        "value": int(
            (~goalscorer_match_coverage["all_scorers_known"]).sum()
        )
    },
    {
        "check": "Matches with all recorded minutes known",
        "value": int(
            goalscorer_match_coverage["all_minutes_known"].sum()
        )
    }
])

display(match_coverage_summary)

,check,value
0,Safe result matches,49481
1,Matches with at least one goalscorer record,15514
2,Matches with at least one known scorer,15508
3,Matches with all recorded scorers known,15508
4,Matches containing missing scorer names,6
5,Matches with all recorded minutes known,15442


## Cell 51 — Compare recorded goal events with match scores

In [ ]:
goalscorer_match_completeness = (
    goalscorer_match_coverage.merge(
        result_question_source[
            [
                "match_id",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized",
                "home_score",
                "away_score"
            ]
        ],
        on="match_id",
        how="left",
        validate="one_to_one"
    )
)

goalscorer_match_completeness["result_total_goals"] = (
    goalscorer_match_completeness["home_score"]
    + goalscorer_match_completeness["away_score"]
)

goalscorer_match_completeness["goal_events_match_score"] = (
    goalscorer_match_completeness["recorded_goal_events"]
    == goalscorer_match_completeness["result_total_goals"]
)

goalscorer_completeness_summary = pd.DataFrame([
    {
        "check": "Matches represented in goalscorer data",
        "value": len(goalscorer_match_completeness)
    },
    {
        "check": "Goal-event count matches final score",
        "value": int(
            goalscorer_match_completeness[
                "goal_events_match_score"
            ].sum()
        )
    },
    {
        "check": "Goal-event count differs from final score",
        "value": int(
            (~goalscorer_match_completeness[
                "goal_events_match_score"
            ]).sum()
        )
    },
    {
        "check": "Complete events and all scorers known",
        "value": int(
            (
                goalscorer_match_completeness[
                    "goal_events_match_score"
                ]
                & goalscorer_match_completeness[
                    "all_scorers_known"
                ]
            ).sum()
        )
    }
])

display(goalscorer_completeness_summary)

,check,value
0,Matches represented in goalscorer data,15514
1,Goal-event count matches final score,15514
2,Goal-event count differs from final score,0
3,Complete events and all scorers known,15508


## Cell 52 — Inspect different scorer-data quality categories

In [ ]:
quality_examples = {
    "Complete events and scorer names": (
        goalscorer_match_completeness[
            goalscorer_match_completeness["goal_events_match_score"]
            & goalscorer_match_completeness["all_scorers_known"]
        ]
    ),
    "Complete events but missing scorer names": (
        goalscorer_match_completeness[
            goalscorer_match_completeness["goal_events_match_score"]
            & ~goalscorer_match_completeness["all_scorers_known"]
        ]
    ),
    "Incomplete goal-event coverage": (
        goalscorer_match_completeness[
            ~goalscorer_match_completeness["goal_events_match_score"]
        ]
    )
}

for category_name, category_df in quality_examples.items():
    print(f"\nCategory: {category_name}")
    print(f"Number of matches: {len(category_df):,}")

    if len(category_df) > 0:
        display(
            category_df[
                [
                    "match_id",
                    "date_standardized",
                    "home_team_standardized",
                    "away_team_standardized",
                    "home_score",
                    "away_score",
                    "recorded_goal_events",
                    "known_scorer_events",
                    "missing_scorer_events"
                ]
            ].sample(
                n=min(5, len(category_df)),
                random_state=42
            )
        )


Category: Complete events and scorer names
Number of matches: 15,508


,match_id,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,recorded_goal_events,known_scorer_events,missing_scorer_events
6338,MATCH_25167,2001-01-28,Togo,Cameroon,0,2,2,2,0
2925,MATCH_13862,1983-09-21,Iceland,Republic of Ireland,0,3,3,3,0
6171,MATCH_24683,2000-06-28,Venezuela,Bolivia,4,2,6,6,0
3595,MATCH_16585,1989-03-10,Yemen,Syria,0,1,1,1,0
10027,MATCH_35615,2012-01-31,Niger,Morocco,0,1,1,1,0



Category: Complete events but missing scorer names
Number of matches: 6


,match_id,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,recorded_goal_events,known_scorer_events,missing_scorer_events
2476,MATCH_12126,1980-02-24,Vanuatu,Papua New Guinea,3,4,7,0,7
2477,MATCH_12127,1980-02-25,Fiji,Solomon Islands,3,1,4,0,4
2486,MATCH_12142,1980-02-29,Fiji,Tahiti,3,6,9,0,9
2480,MATCH_12132,1980-02-26,New Caledonia,Vanuatu,4,3,7,0,7
2485,MATCH_12141,1980-02-28,New Caledonia,Papua New Guinea,8,0,8,0,8



Category: Incomplete goal-event coverage
Number of matches: 0


## 7.4 Goalscorer Data Coverage

The final goalscorer dataset contains 47,839 safely linked goal-event records covering 15,514 matches.

Validation produced the following findings:

- Every represented match has a recorded goal-event count equal to its final total score.
- 15,508 matches have names for every recorded scorer.
- Six matches contain missing scorer names.
- All 48 missing scorer values occur within those six matches.
- 15,442 matches have minutes recorded for every goal event.
- Therefore, 72 matches contain at least one missing scoring minute.
- The dataset includes 926 own-goal events.
- The dataset includes 3,259 penalty-goal events.

The 15,508 matches with complete goal-event coverage and complete scorer names are reliable enough for scorer-list questions.

Matches absent from the goalscorer dataset will not be used for scorer-based question generation. Their absence does not prove that no goals were scored because historical goalscorer coverage is incomplete.

The six matches containing missing scorer names will also be excluded from scorer-list questions because a complete answer cannot be produced without inventing information.

## Cell 54 — Create the reliable scorer-question source

In [ ]:
reliable_scorer_match_ids = set(
    goalscorer_match_completeness.loc[
        goalscorer_match_completeness["goal_events_match_score"]
        & goalscorer_match_completeness["all_scorers_known"],
        "match_id"
    ]
)

reliable_scorer_events = goalscorer_question_source[
    goalscorer_question_source["match_id"].isin(
        reliable_scorer_match_ids
    )
].copy()

print(
    "Reliable matches for complete scorer questions:",
    f"{len(reliable_scorer_match_ids):,}"
)
print(
    "Goal events in reliable matches:",
    f"{len(reliable_scorer_events):,}"
)
print(
    "Missing scorer names:",
    f"{reliable_scorer_events['scorer'].isna().sum():,}"
)

assert len(reliable_scorer_match_ids) == 15_508
assert reliable_scorer_events["scorer"].notna().all()

print("Reliable scorer-question source created.")

Reliable matches for complete scorer questions: 15,508
Goal events in reliable matches: 47,791
Missing scorer names: 0
Reliable scorer-question source created.


## Cell 55 — Inspect own-goal and penalty formatting fields

In [ ]:
special_goal_examples = reliable_scorer_events[
    reliable_scorer_events["own_goal"].fillna(False)
    | reliable_scorer_events["penalty"].fillna(False)
][
    [
        "match_id",
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "team",
        "scorer",
        "minute_original",
        "own_goal",
        "penalty"
    ]
].sample(
    n=10,
    random_state=42
)

display(special_goal_examples)

,match_id,date_standardized,home_team_standardized,away_team_standardized,team,scorer,minute_original,own_goal,penalty
38310,MATCH_42423,2019-03-26,Norway,Sweden,Sweden,Håvard Nordtveit,86,True,False
43593,MATCH_46441,2023-09-12,Venezuela,Paraguay,Venezuela,Salomón Rondón,90,False,True
37776,MATCH_41801,2018-09-09,Ukraine,Slovakia,Ukraine,Andriy Yarmolenko,80,False,True
2195,MATCH_03901,1954-04-03,Haiti,United States,Haiti,Gerard Ellie,60,False,True
31624,MATCH_35603,2012-01-26,Sudan,Angola,Angola,Manucho,50,False,True
2369,MATCH_04052,1955-03-06,Chile,Peru,Peru,Cornelio Heredia,63,False,True
37802,MATCH_41842,2018-09-11,Iceland,Belgium,Belgium,Eden Hazard,29,False,True
11775,MATCH_16466,1988-12-09,Saudi Arabia,Bahrain,Bahrain,Fayad Mahmoud,44,False,True
14616,MATCH_19379,1993-10-13,Bulgaria,Austria,Bulgaria,Hristo Stoichkov,33,False,True
25213,MATCH_29194,2005-06-04,Chile,Bolivia,Bolivia,José Alfredo Castillo,83,False,True


## Cell 56 — Format normal, penalty, and own-goal events safely

In [ ]:
def format_goal_event(row):
    scorer = str(row["scorer"]).strip()
    credited_team = str(row["team"]).strip()

    is_own_goal = row["own_goal"] is True
    is_penalty = row["penalty"] is True

    if is_own_goal:
        return (
            f"{scorer} "
            f"(own goal credited to {credited_team})"
        )

    if is_penalty:
        return (
            f"{scorer} for {credited_team} "
            f"(penalty)"
        )

    return f"{scorer} for {credited_team}"


reliable_scorer_events["formatted_goal_event"] = (
    reliable_scorer_events.apply(
        format_goal_event,
        axis=1
    )
)

format_validation_examples = pd.concat(
    [
        reliable_scorer_events[
            reliable_scorer_events["own_goal"].eq(True)
        ].sample(
            n=min(
                5,
                reliable_scorer_events["own_goal"].eq(True).sum()
            ),
            random_state=42
        ),
        reliable_scorer_events[
            reliable_scorer_events["penalty"].eq(True)
            & ~reliable_scorer_events["own_goal"].eq(True)
        ].sample(
            n=min(
                5,
                (
                    reliable_scorer_events["penalty"].eq(True)
                    & ~reliable_scorer_events["own_goal"].eq(True)
                ).sum()
            ),
            random_state=42
        )
    ],
    ignore_index=True
)

display(
    format_validation_examples[
        [
            "scorer",
            "team",
            "own_goal",
            "penalty",
            "formatted_goal_event"
        ]
    ]
)

,scorer,team,own_goal,penalty,formatted_goal_event
0,Alexander Riley,Trinidad and Tobago,True,False,Alexander Riley (own goal credited to Trinidad and Tobago)
1,Oghenekaro Etebo,Croatia,True,False,Oghenekaro Etebo (own goal credited to Croatia)
2,Benjamín Velasco,Panama,True,False,Benjamín Velasco (own goal credited to Panama)
3,Roger Aholou,South Sudan,True,False,Roger Aholou (own goal credited to South Sudan)
4,Ludovic Magnin,Republic of Ireland,True,False,Ludovic Magnin (own goal credited to Republic of Ireland)
5,Frédéric Kanouté,Mali,False,True,Frédéric Kanouté for Mali (penalty)
6,José Sanfilippo,Argentina,False,True,José Sanfilippo for Argentina (penalty)
7,Davor Šuker,Croatia,False,True,Davor Šuker for Croatia (penalty)
8,Emil Forsberg,Sweden,False,True,Emil Forsberg for Sweden (penalty)
9,Harry Kane,England,False,True,Harry Kane for England (penalty)


## Cell 57 — Aggregate scorer events by match

In [ ]:
def join_goal_events(values):
    event_list = list(values)

    if len(event_list) == 1:
        return event_list[0]

    if len(event_list) == 2:
        return f"{event_list[0]} and {event_list[1]}"

    return (
        ", ".join(event_list[:-1])
        + f", and {event_list[-1]}"
    )


scorer_list_source = (
    reliable_scorer_events
    .groupby("match_id", sort=False)
    .agg(
        scorer_list=(
            "formatted_goal_event",
            join_goal_events
        ),
        recorded_goal_events=("match_id", "size")
    )
    .reset_index()
    .merge(
        result_question_source[
            [
                "match_id",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized",
                "home_score",
                "away_score",
                "tournament"
            ]
        ],
        on="match_id",
        how="left",
        validate="one_to_one"
    )
)

scorer_list_source["result_total_goals"] = (
    scorer_list_source["home_score"]
    + scorer_list_source["away_score"]
)

assert len(scorer_list_source) == 15_508
assert (
    scorer_list_source["recorded_goal_events"]
    == scorer_list_source["result_total_goals"]
).all()

print(
    "Reliable scorer-list matches prepared:",
    f"{len(scorer_list_source):,}"
)

display(
    scorer_list_source[
        [
            "match_id",
            "home_team_standardized",
            "away_team_standardized",
            "home_score",
            "away_score",
            "scorer_list"
        ]
    ].sample(
        n=10,
        random_state=42
    )
)

Reliable scorer-list matches prepared: 15,508


,match_id,home_team_standardized,away_team_standardized,home_score,away_score,scorer_list
6332,MATCH_25167,Togo,Cameroon,0,2,Samuel Eto'o for Cameroon and Patrick M'Boma for Cameroon
2919,MATCH_13862,Iceland,Republic of Ireland,0,3,"Gary Waddock for Republic of Ireland, Michael Robinson for Republic of Ireland, and Mickey Walsh for Republic of Ireland"
6165,MATCH_24683,Venezuela,Bolivia,4,2,"Miguel Mea Vitali for Venezuela, Ruberth Morán for Venezuela, Jaime Moreno for Bolivia, Julio César Baldivieso for Bolivia, Giovanni Savarese for ..."
3589,MATCH_16585,Yemen,Syria,0,1,Nizar Mahrous for Syria
10021,MATCH_35615,Niger,Morocco,0,1,Younès Belhanda for Morocco
6350,MATCH_25225,Kuwait,Singapore,1,0,Khalaf Al-Mutairi for Kuwait
14529,MATCH_47344,Kuwait,Afghanistan,1,0,Eid Al Rashidi for Kuwait
1768,MATCH_09187,New Zealand,Tahiti,1,1,Alan Vest for New Zealand and Erroll Bennett for Tahiti
12334,MATCH_42501,Ukraine,Serbia,5,0,"Viktor Tsyhankov for Ukraine, Viktor Tsyhankov for Ukraine, Yevhen Konoplyanka for Ukraine, Roman Yaremchuk for Ukraine, and Yevhen Konoplyanka fo..."
7934,MATCH_29314,United States,Cuba,4,1,"Lester Moré for Cuba, Clint Dempsey for United States, Landon Donovan for United States, Landon Donovan for United States, and DaMarcus Beasley fo..."


## Cell 58 — Generate scorer-list questions

In [ ]:
scorer_list_qa_df = scorer_list_source.copy()

scorer_list_qa_df["question_id"] = [
    f"SCORERS_{number:05d}"
    for number in range(1, len(scorer_list_qa_df) + 1)
]

scorer_list_qa_df["intent"] = "match_scorers"

scorer_list_qa_df["question"] = (
    "Who scored in the match between "
    + scorer_list_qa_df["home_team_standardized"]
    + " and "
    + scorer_list_qa_df["away_team_standardized"]
    + " on "
    + scorer_list_qa_df["date_standardized"]
    + "?"
)

scorer_list_qa_df["answer"] = (
    "The recorded scorers were "
    + scorer_list_qa_df["scorer_list"]
    + "."
)

scorer_list_qa_df = scorer_list_qa_df[
    [
        "question_id",
        "match_id",
        "intent",
        "question",
        "answer",
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "home_score",
        "away_score",
        "tournament"
    ]
]

print(
    "Scorer-list questions generated:",
    f"{len(scorer_list_qa_df):,}"
)

assert len(scorer_list_qa_df) == 15_508
assert scorer_list_qa_df["question_id"].is_unique
assert scorer_list_qa_df["question"].is_unique
assert scorer_list_qa_df["answer"].notna().all()

print("Scorer-list question validation passed.")

display(
    scorer_list_qa_df[
        [
            "question_id",
            "question",
            "answer",
            "tournament"
        ]
    ].sample(
        n=10,
        random_state=42
    )
)

Scorer-list questions generated: 15,508
Scorer-list question validation passed.


,question_id,question,answer,tournament
6332,SCORERS_06333,Who scored in the match between Togo and Cameroon on 2001-01-28?,The recorded scorers were Samuel Eto'o for Cameroon and Patrick M'Boma for Cameroon.,FIFA World Cup qualification
2919,SCORERS_02920,Who scored in the match between Iceland and Republic of Ireland on 1983-09-21?,"The recorded scorers were Gary Waddock for Republic of Ireland, Michael Robinson for Republic of Ireland, and Mickey Walsh for Republic of Ireland.",UEFA Euro qualification
6165,SCORERS_06166,Who scored in the match between Venezuela and Bolivia on 2000-06-28?,"The recorded scorers were Miguel Mea Vitali for Venezuela, Ruberth Morán for Venezuela, Jaime Moreno for Bolivia, Julio César Baldivieso for Boliv...",FIFA World Cup qualification
3589,SCORERS_03590,Who scored in the match between Yemen and Syria on 1989-03-10?,The recorded scorers were Nizar Mahrous for Syria.,FIFA World Cup qualification
10021,SCORERS_10022,Who scored in the match between Niger and Morocco on 2012-01-31?,The recorded scorers were Younès Belhanda for Morocco.,African Cup of Nations
6350,SCORERS_06351,Who scored in the match between Kuwait and Singapore on 2001-02-21?,The recorded scorers were Khalaf Al-Mutairi for Kuwait.,FIFA World Cup qualification
14529,SCORERS_14530,Who scored in the match between Kuwait and Afghanistan on 2024-06-11?,The recorded scorers were Eid Al Rashidi for Kuwait.,FIFA World Cup qualification
1768,SCORERS_01769,Who scored in the match between New Zealand and Tahiti on 1973-02-18?,The recorded scorers were Alan Vest for New Zealand and Erroll Bennett for Tahiti.,Oceania Nations Cup
12334,SCORERS_12335,Who scored in the match between Ukraine and Serbia on 2019-06-07?,"The recorded scorers were Viktor Tsyhankov for Ukraine, Viktor Tsyhankov for Ukraine, Yevhen Konoplyanka for Ukraine, Roman Yaremchuk for Ukraine,...",UEFA Euro qualification
7934,SCORERS_07935,Who scored in the match between United States and Cuba on 2005-07-07?,"The recorded scorers were Lester Moré for Cuba, Clint Dempsey for United States, Landon Donovan for United States, Landon Donovan for United State...",Gold Cup


## Cell 59 — Validate the regenerated scorer-list dataset

In [ ]:
scorer_list_validation = pd.DataFrame([
    {
        "check": "Scorer-list questions",
        "value": len(scorer_list_qa_df)
    },
    {
        "check": "Duplicate question IDs",
        "value": int(
            scorer_list_qa_df["question_id"].duplicated().sum()
        )
    },
    {
        "check": "Duplicate question text",
        "value": int(
            scorer_list_qa_df["question"].duplicated().sum()
        )
    },
    {
        "check": "Missing answers",
        "value": int(
            scorer_list_qa_df["answer"].isna().sum()
        )
    },
    {
        "check": "Answers containing own-goal labels",
        "value": int(
            scorer_list_qa_df["answer"]
            .str.contains(
                "own goal credited to",
                regex=False,
                na=False
            )
            .sum()
        )
    },
    {
        "check": "Answers containing penalty labels",
        "value": int(
            scorer_list_qa_df["answer"]
            .str.contains(
                "(penalty)",
                regex=False,
                na=False
            )
            .sum()
        )
    }
])

display(scorer_list_validation)

assert len(scorer_list_qa_df) == 15_508
assert scorer_list_qa_df["question_id"].is_unique
assert scorer_list_qa_df["question"].is_unique
assert scorer_list_qa_df["answer"].notna().all()

print("Regenerated scorer-list dataset validated.")

,check,value
0,Scorer-list questions,15508
1,Duplicate question IDs,0
2,Duplicate question text,0
3,Missing answers,0
4,Answers containing own-goal labels,896
5,Answers containing penalty labels,2893


Regenerated scorer-list dataset validated.


## Cell 60 — Prepare ordinary player-goal events

In [ ]:
player_goal_events = reliable_scorer_events[
    ~reliable_scorer_events["own_goal"].eq(True)
].copy()

print(
    "Ordinary player-goal events:",
    f"{len(player_goal_events):,}"
)
print(
    "Own-goal events excluded:",
    f"{reliable_scorer_events['own_goal'].eq(True).sum():,}"
)
print(
    "Penalty goals retained:",
    f"{player_goal_events['penalty'].eq(True).sum():,}"
)

assert not player_goal_events["own_goal"].eq(True).any()
assert player_goal_events["scorer"].notna().all()

print("Player-goal source prepared.")

Ordinary player-goal events: 46,865
Own-goal events excluded: 926
Penalty goals retained: 3,259
Player-goal source prepared.


# Cell 61 — Aggregate each player’s goals within a match

In [ ]:
player_match_goal_source = (
    player_goal_events
    .groupby(
        [
            "match_id",
            "scorer",
            "team"
        ],
        sort=False,
        dropna=False
    )
    .agg(
        player_goal_count=("scorer", "size"),
        penalty_goal_count=(
            "penalty",
            lambda values: values.eq(True).sum()
        )
    )
    .reset_index()
    .merge(
        result_question_source[
            [
                "match_id",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized",
                "home_score",
                "away_score",
                "tournament"
            ]
        ],
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

print(
    "Player-within-match goal-count records:",
    f"{len(player_match_goal_source):,}"
)

display(
    player_match_goal_source[
        [
            "match_id",
            "scorer",
            "team",
            "player_goal_count",
            "penalty_goal_count",
            "home_team_standardized",
            "away_team_standardized",
            "date_standardized"
        ]
    ].sort_values(
        "player_goal_count",
        ascending=False
    ).head(15)
)

Player-within-match goal-count records: 39,931


,match_id,scorer,team,player_goal_count,penalty_goal_count,home_team_standardized,away_team_standardized,date_standardized
17029,MATCH_25424,Archie Thompson,Australia,13,0,Australia,American Samoa,2001-04-11
17030,MATCH_25424,David Zdrilic,Australia,8,0,Australia,American Samoa,2001-04-11
7496,MATCH_12900,Gary Cole,Australia,7,1,Australia,Fiji,1981-08-14
11759,MATCH_19243,Luís Roberto Alves,Mexico,7,0,Mexico,Martinique,1993-07-11
14077,MATCH_21995,Karim Bagheri,Iran,7,0,Maldives,Iran,1997-06-02
16692,MATCH_25082,Karim Bagheri,Iran,6,0,Iran,Guam,2000-11-24
17021,MATCH_25421,John Aloisi,Australia,6,0,Australia,Tonga,2001-04-09
6250,MATCH_10966,Hans Krankl,Austria,6,0,Austria,Malta,1977-04-30
7498,MATCH_12905,Steve Sumner,New Zealand,6,0,New Zealand,Fiji,1981-08-16
14306,MATCH_22111,Kazuyoshi Miura,Japan,6,0,Japan,Macau,1997-06-22


## Cell 62 — Generate player goal-count questions

In [ ]:
def create_player_goal_count_answer(row):
    scorer = row["scorer"]
    goal_count = int(row["player_goal_count"])
    team = row["team"]

    goal_word = "goal" if goal_count == 1 else "goals"

    return (
        f"{scorer} scored {goal_count} {goal_word} "
        f"for {team} in the match."
    )


player_goal_count_qa_df = player_match_goal_source.copy()

player_goal_count_qa_df["question_id"] = [
    f"PGCOUNT_{number:05d}"
    for number in range(
        1,
        len(player_goal_count_qa_df) + 1
    )
]

player_goal_count_qa_df["intent"] = "player_match_goal_count"

player_goal_count_qa_df["question"] = (
    "How many goals did "
    + player_goal_count_qa_df["scorer"]
    + " score in the match between "
    + player_goal_count_qa_df["home_team_standardized"]
    + " and "
    + player_goal_count_qa_df["away_team_standardized"]
    + " on "
    + player_goal_count_qa_df["date_standardized"]
    + "?"
)

player_goal_count_qa_df["answer"] = (
    player_goal_count_qa_df.apply(
        create_player_goal_count_answer,
        axis=1
    )
)

player_goal_count_qa_df = player_goal_count_qa_df[
    [
        "question_id",
        "match_id",
        "intent",
        "question",
        "answer",
        "scorer",
        "team",
        "player_goal_count",
        "date_standardized",
        "home_team_standardized",
        "away_team_standardized",
        "home_score",
        "away_score",
        "tournament"
    ]
]

assert player_goal_count_qa_df["question_id"].is_unique
assert player_goal_count_qa_df["question"].is_unique
assert player_goal_count_qa_df["answer"].notna().all()

print(
    "Player goal-count questions generated:",
    f"{len(player_goal_count_qa_df):,}"
)
print("Player goal-count validation passed.")

display(
    player_goal_count_qa_df[
        [
            "question_id",
            "question",
            "answer",
            "tournament"
        ]
    ].sample(
        n=10,
        random_state=42
    )
)

Player goal-count questions generated: 39,931
Player goal-count validation passed.


,question_id,question,answer,tournament
15529,PGCOUNT_15530,How many goals did Miguel Zepeda score in the match between Chile and Mexico on 1999-07-17?,Miguel Zepeda scored 1 goal for Mexico in the match.,Copa América
14240,PGCOUNT_14241,How many goals did Marco Etcheverry score in the match between Bolivia and Peru on 1997-06-15?,Marco Etcheverry scored 1 goal for Bolivia in the match.,Copa América
16971,PGCOUNT_16972,How many goals did Ümit Davala score in the match between North Macedonia and Turkey on 2001-03-28?,Ümit Davala scored 1 goal for Turkey in the match.,FIFA World Cup qualification
38977,PGCOUNT_38978,How many goals did Georges Mikautadze score in the match between Georgia and Bulgaria on 2025-09-07?,Georges Mikautadze scored 1 goal for Georgia in the match.,FIFA World Cup qualification
30759,PGCOUNT_30760,How many goals did Christian Eriksen score in the match between Denmark and Poland on 2017-09-01?,Christian Eriksen scored 1 goal for Denmark in the match.,FIFA World Cup qualification
24778,PGCOUNT_24779,How many goals did David Villa score in the match between Paraguay and Spain on 2010-07-03?,David Villa scored 1 goal for Spain in the match.,FIFA World Cup
29502,PGCOUNT_29503,How many goals did Sebastián Soria score in the match between Qatar and Hong Kong on 2016-03-24?,Sebastián Soria scored 1 goal for Qatar in the match.,FIFA World Cup qualification
26353,PGCOUNT_26354,How many goals did Rémy Ebanega score in the match between Gabon and Burkina Faso on 2012-06-09?,Rémy Ebanega scored 1 goal for Gabon in the match.,FIFA World Cup qualification
34134,PGCOUNT_34135,How many goals did Eran Zahavi score in the match between Faroe Islands and Israel on 2021-09-01?,Eran Zahavi scored 3 goals for Israel in the match.,FIFA World Cup qualification
8304,PGCOUNT_08305,How many goals did Eddy Voordeckers score in the match between Belgium and Albania on 1984-10-17?,Eddy Voordeckers scored 1 goal for Belgium in the match.,FIFA World Cup qualification


## 7.5 Scorer-List and Player Goal-Count Questions

Scorer-list questions were generated for the 15,508 matches having complete goal-event coverage and known scorer names.

Special goal events were handled explicitly:

- Penalty goals remain credited to the scorer and are labelled as penalties.
- Own goals are described using the form `Player (own goal credited to Team)`.
- This avoids incorrectly implying that an own-goal scorer represented the team credited with the goal.
- Repeated scorer names are retained in scorer-list answers because each occurrence represents a separate goal event.

This produced 15,508 validated `match_scorers` questions.

For player goal-count questions, all 926 own-goal events were excluded because an own goal should not count as a goal scored for the named player. Penalty goals remained included.

The remaining 46,865 ordinary player-goal events were grouped by match, player, and team. This produced 39,931 validated `player_match_goal_count` questions.

Grouping ensures that a player who scored multiple times in one match receives one question containing the correct total rather than several duplicate questions.

## Cell 64 — Validate player goal-count aggregation

In [ ]:
player_goal_count_validation = pd.DataFrame([
    {
        "check": "Ordinary goal events",
        "value": len(player_goal_events)
    },
    {
        "check": "Player-within-match records",
        "value": len(player_match_goal_source)
    },
    {
        "check": "Generated goal-count questions",
        "value": len(player_goal_count_qa_df)
    },
    {
        "check": "Sum of aggregated goal counts",
        "value": int(
            player_match_goal_source[
                "player_goal_count"
            ].sum()
        )
    },
    {
        "check": "Duplicate question IDs",
        "value": int(
            player_goal_count_qa_df[
                "question_id"
            ].duplicated().sum()
        )
    },
    {
        "check": "Duplicate question text",
        "value": int(
            player_goal_count_qa_df[
                "question"
            ].duplicated().sum()
        )
    },
    {
        "check": "Missing answers",
        "value": int(
            player_goal_count_qa_df[
                "answer"
            ].isna().sum()
        )
    }
])

display(player_goal_count_validation)

assert (
    player_match_goal_source["player_goal_count"].sum()
    == len(player_goal_events)
)

assert (
    len(player_match_goal_source)
    == len(player_goal_count_qa_df)
)

print("Player goal-count aggregation fully validated.")

,check,value
0,Ordinary goal events,46865
1,Player-within-match records,39931
2,Generated goal-count questions,39931
3,Sum of aggregated goal counts,46865
4,Duplicate question IDs,0
5,Duplicate question text,0
6,Missing answers,0


Player goal-count aggregation fully validated.


## Cell 65 — Inspect scoring-minute storage

In [ ]:
print(
    "minute_original data type:",
    reliable_scorer_events["minute_original"].dtype
)

minute_storage_summary = pd.DataFrame([
    {
        "check": "Reliable goal events",
        "value": len(reliable_scorer_events)
    },
    {
        "check": "Known minute values",
        "value": int(
            reliable_scorer_events[
                "minute_original"
            ].notna().sum()
        )
    },
    {
        "check": "Missing minute values",
        "value": int(
            reliable_scorer_events[
                "minute_original"
            ].isna().sum()
        )
    },
    {
        "check": "Known ordinary-goal minutes",
        "value": int(
            player_goal_events[
                "minute_original"
            ].notna().sum()
        )
    },
    {
        "check": "Missing ordinary-goal minutes",
        "value": int(
            player_goal_events[
                "minute_original"
            ].isna().sum()
        )
    }
])

display(minute_storage_summary)

display(
    reliable_scorer_events[
        [
            "scorer",
            "team",
            "minute_original",
            "own_goal",
            "penalty",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized"
        ]
    ].sample(
        n=15,
        random_state=42
    )
)

minute_original data type: object


,check,value
0,Reliable goal events,47791
1,Known minute values,47583
2,Missing minute values,208
3,Known ordinary-goal minutes,46657
4,Missing ordinary-goal minutes,208


,scorer,team,minute_original,own_goal,penalty,date_standardized,home_team_standardized,away_team_standardized
44115,Bertrand Traoré,Burkina Faso,78,False,True,2023-11-21,Ethiopia,Burkina Faso
29938,Tiago Mendes,Portugal,89,False,False,2010-06-21,Portugal,North Korea
2574,Antonio Valentín Angelillo,Argentina,39,False,False,1957-03-17,Argentina,Ecuador
36901,Raúl Jiménez,Mexico,54,False,False,2017-06-21,Mexico,New Zealand
43231,Daniel James,Wales,10,False,False,2023-06-16,Wales,Armenia
10552,Panayiotis Marangos,Cyprus,8,False,False,1985-02-27,Netherlands,Cyprus
41271,Jim Allevinah,Gabon,73,False,False,2021-09-05,Gabon,Egypt
10871,Julio César Romero,Paraguay,40,False,False,1985-06-23,Brazil,Paraguay
26984,Steffen Iversen,Norway,49,False,False,2007-09-08,Moldova,Norway
13315,Paul Wade,Australia,67,False,False,1992-09-26,Australia,Solomon Islands


## Cell 66 — Inspect unique minute formats and ranges

In [ ]:
known_minute_values = (
    reliable_scorer_events.loc[
        reliable_scorer_events["minute_original"].notna(),
        "minute_original"
    ]
    .astype(str)
    .str.strip()
)

valid_minute_pattern = (
    known_minute_values.str.fullmatch(
        r"\d+|\d+\.0|\d+\+\d+"
    )
)

minute_pattern_summary = pd.DataFrame([
    {
        "check": "Values containing +",
        "value": int(
            known_minute_values.str.contains(
                "+",
                regex=False
            ).sum()
        )
    },
    {
        "check": "Integer-like values",
        "value": int(
            known_minute_values.str.fullmatch(
                r"\d+"
            ).sum()
        )
    },
    {
        "check": "Decimal-like values",
        "value": int(
            known_minute_values.str.fullmatch(
                r"\d+\.0"
            ).sum()
        )
    },
    {
        "check": "Other formats",
        "value": int(
            (~valid_minute_pattern).sum()
        )
    }
])

display(minute_pattern_summary)

assert valid_minute_pattern.all()

print("All known minute values use supported formats.")

,check,value
0,Values containing +,16
1,Integer-like values,47567
2,Decimal-like values,0
3,Other formats,0


All known minute values use supported formats.


## Cell 67 — Inspect unusual and missing minute records

In [ ]:
unusual_minute_mask = (
    reliable_scorer_events["minute_original"].notna()
    & ~reliable_scorer_events[
        "minute_original"
    ].astype(str).str.strip().str.fullmatch(
        r"\d+|\d+\.0|\d+\+\d+"
    )
)

print(
    "Unusual non-missing minute records:",
    f"{unusual_minute_mask.sum():,}"
)

if unusual_minute_mask.any():
    display(
        reliable_scorer_events.loc[
            unusual_minute_mask,
            [
                "match_id",
                "scorer",
                "team",
                "minute_original",
                "own_goal",
                "penalty",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized"
            ]
        ].head(30)
    )

print(
    "\nReliable events with missing minutes:",
    f"{reliable_scorer_events['minute_original'].isna().sum():,}"
)

display(
    reliable_scorer_events.loc[
        reliable_scorer_events[
            "minute_original"
        ].isna(),
        [
            "match_id",
            "scorer",
            "team",
            "minute_original",
            "own_goal",
            "penalty",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized"
        ]
    ].head(20)
)

Unusual non-missing minute records: 0

Reliable events with missing minutes: 208


,match_id,scorer,team,minute_original,own_goal,penalty,date_standardized,home_team_standardized,away_team_standardized
3347,MATCH_05164,Yiu Cheuk Yin,Taiwan,<NA>,False,False,1960-10-16,Taiwan,Vietnam Republic
4059,MATCH_05974,Edward Acquah,Ghana,<NA>,False,False,1963-11-26,Ghana,Ethiopia
4060,MATCH_05974,Edward Acquah,Ghana,<NA>,False,False,1963-11-26,Ghana,Ethiopia
4063,MATCH_05976,Mengistu Worku,Ethiopia,<NA>,False,False,1963-11-28,Ethiopia,Tunisia
4064,MATCH_05976,Mengistu Worku,Ethiopia,<NA>,False,False,1963-11-28,Ethiopia,Tunisia
4065,MATCH_05976,Girma Tekle,Ethiopia,<NA>,False,False,1963-11-28,Ethiopia,Tunisia
4066,MATCH_05976,Girma Tesfaye,Ethiopia,<NA>,False,False,1963-11-28,Ethiopia,Tunisia
4067,MATCH_05977,Nasr Eddin Abbas,Sudan,<NA>,False,False,1963-11-28,Nigeria,Sudan
4068,MATCH_05977,Nasr Eddin Abbas,Sudan,<NA>,False,False,1963-11-28,Nigeria,Sudan
4069,MATCH_05977,Ibrahim Yahia El-Kawarty,Sudan,<NA>,False,False,1963-11-28,Nigeria,Sudan


## 7.6 Scoring-Minute Data Inspection

Scoring-minute inspection was performed on the 47,791 reliable goal events having known scorer names.

The inspection found:

- 47,583 events have known scoring minutes.
- 208 events have missing scoring minutes.
- All 208 missing-minute events are ordinary player-goal events rather than own goals.
- Known minutes use integer or added-time formats such as `45+2`, `90+4`, and `120+1`.
- No unsupported non-missing minute formats were found.
- Added-time values must remain as text so their original meaning is preserved.

Scoring-minute questions will exclude own goals because they are not ordinary goals scored for the named player.

A player-within-match record will be used only when the minutes of all ordinary goals scored by that player in that match are known. This prevents partially complete answers from being presented as complete.

## Cell 69 — Check minute completeness per player and match

In [ ]:
player_minute_completeness = (
    player_goal_events
    .groupby(
        [
            "match_id",
            "scorer",
            "team"
        ],
        sort=False,
        dropna=False
    )
    .agg(
        total_player_goals=("scorer", "size"),
        known_minute_count=(
            "minute_original",
            lambda values: values.notna().sum()
        ),
        missing_minute_count=(
            "minute_original",
            lambda values: values.isna().sum()
        )
    )
    .reset_index()
)

player_minute_completeness["all_minutes_known"] = (
    player_minute_completeness["missing_minute_count"].eq(0)
)

minute_group_summary = pd.DataFrame([
    {
        "check": "Player-within-match records",
        "value": len(player_minute_completeness)
    },
    {
        "check": "Records with all minutes known",
        "value": int(
            player_minute_completeness[
                "all_minutes_known"
            ].sum()
        )
    },
    {
        "check": "Records containing missing minutes",
        "value": int(
            (~player_minute_completeness[
                "all_minutes_known"
            ]).sum()
        )
    },
    {
        "check": "Missing ordinary-goal minute events",
        "value": int(
            player_minute_completeness[
                "missing_minute_count"
            ].sum()
        )
    }
])

display(minute_group_summary)

assert (
    player_minute_completeness[
        "missing_minute_count"
    ].sum()
    == 208
)

,check,value
0,Player-within-match records,39931
1,Records with all minutes known,39758
2,Records containing missing minutes,173
3,Missing ordinary-goal minute events,208


## Cell 70 — Create the reliable scoring-minute events

In [ ]:
reliable_minute_groups = (
    player_minute_completeness.loc[
        player_minute_completeness["all_minutes_known"],
        [
            "match_id",
            "scorer",
            "team"
        ]
    ]
)

reliable_minute_events = (
    player_goal_events
    .merge(
        reliable_minute_groups,
        on=[
            "match_id",
            "scorer",
            "team"
        ],
        how="inner",
        validate="many_to_one"
    )
    .copy()
)

assert reliable_minute_events["minute_original"].notna().all()
assert not reliable_minute_events["own_goal"].eq(True).any()

print(
    "Reliable scoring-minute events:",
    f"{len(reliable_minute_events):,}"
)

print(
    "Reliable player-within-match minute records:",
    f"{len(reliable_minute_groups):,}"
)

print("Reliable scoring-minute source created.")

Reliable scoring-minute events: 46,655
Reliable player-within-match minute records: 39,758
Reliable scoring-minute source created.


## Cell 71 — Normalize and order scoring-minute values

In [ ]:
def clean_minute_text(value):
    minute_text = str(value).strip()

    if minute_text.endswith(".0"):
        minute_text = minute_text[:-2]

    return minute_text


def minute_sort_value(value):
    minute_text = clean_minute_text(value)

    if "+" in minute_text:
        base_minute, added_minute = minute_text.split(
            "+",
            maxsplit=1
        )

        return int(base_minute) + int(added_minute) / 100

    return int(minute_text)


reliable_minute_events["minute_text"] = (
    reliable_minute_events[
        "minute_original"
    ].apply(clean_minute_text)
)

reliable_minute_events["minute_sort_value"] = (
    reliable_minute_events[
        "minute_original"
    ].apply(minute_sort_value)
)

assert (
    reliable_minute_events["minute_text"]
    .str.fullmatch(r"\d+|\d+\+\d+")
    .all()
)

display(
    reliable_minute_events[
        [
            "scorer",
            "team",
            "minute_original",
            "minute_text",
            "minute_sort_value"
        ]
    ].sort_values(
        "minute_sort_value"
    ).tail(20)
)

,scorer,team,minute_original,minute_text,minute_sort_value
25617,Fabio Grosso,Italy,119,119,119.00
27500,Ivan Klasnić,Croatia,119,119,119.00
7349,Hassan Rowshan,Iran,119,119,119.00
9943,Michel Platini,France,119,119,119.00
25618,Alessandro Del Piero,Italy,120,120,120.00
27501,Semih Şentürk,Turkey,120,120,120.00
39993,Artem Dovbyk,Ukraine,120,120,120.00
22791,Jaouad Zairi,Morocco,120,120,120.00
43350,Oumar Diakité,Ivory Coast,120,120,120.00
29654,Hwang Jae-won,South Korea,120,120,120.00


## Cell 72 — Aggregate scoring minutes by player and match

In [ ]:
def join_scoring_minutes(values):
    minute_list = [
        f"{value}'"
        for value in values
    ]

    if len(minute_list) == 1:
        return minute_list[0]

    if len(minute_list) == 2:
        return (
            f"{minute_list[0]} and "
            f"{minute_list[1]}"
        )

    return (
        ", ".join(minute_list[:-1])
        + f", and {minute_list[-1]}"
    )


player_match_minute_source = (
    reliable_minute_events
    .sort_values(
        [
            "match_id",
            "scorer",
            "team",
            "minute_sort_value"
        ]
    )
    .groupby(
        [
            "match_id",
            "scorer",
            "team"
        ],
        sort=False,
        dropna=False
    )
    .agg(
        scoring_minutes=(
            "minute_text",
            join_scoring_minutes
        ),
        goal_count=("scorer", "size")
    )
    .reset_index()
    .merge(
        result_question_source[
            [
                "match_id",
                "date_standardized",
                "home_team_standardized",
                "away_team_standardized",
                "home_score",
                "away_score",
                "tournament"
            ]
        ],
        on="match_id",
        how="left",
        validate="many_to_one"
    )
)

assert (
    player_match_minute_source["goal_count"].sum()
    == len(reliable_minute_events)
)

assert (
    len(player_match_minute_source)
    == len(reliable_minute_groups)
)

print(
    "Player-within-match minute records:",
    f"{len(player_match_minute_source):,}"
)

display(
    player_match_minute_source[
        [
            "match_id",
            "scorer",
            "team",
            "goal_count",
            "scoring_minutes",
            "home_team_standardized",
            "away_team_standardized",
            "date_standardized"
        ]
    ].sort_values(
        "goal_count",
        ascending=False
    ).head(15)
)

Player-within-match minute records: 39,758


,match_id,scorer,team,goal_count,scoring_minutes,home_team_standardized,away_team_standardized,date_standardized
16855,MATCH_25424,Archie Thompson,Australia,13,"12', 23', 27', 29', 32', 37', 42', 45', 56', 60', 65', 85', and 88'",Australia,American Samoa,2001-04-11
16858,MATCH_25424,David Zdrilic,Australia,8,"13', 21', 25', 33', 58', 66', 78', and 89'",Australia,American Samoa,2001-04-11
13906,MATCH_21995,Karim Bagheri,Iran,7,"9', 13', 16', 60', 66', 67', and 86'",Maldives,Iran,1997-06-02
7353,MATCH_12900,Gary Cole,Australia,7,"35', 52', 64', 67', 71', 75', and 78'",Australia,Fiji,1981-08-14
11594,MATCH_19243,Luís Roberto Alves,Mexico,7,"11', 21', 29', 54', 76', 84', and 90'",Mexico,Martinique,1993-07-11
6153,MATCH_10966,Hans Krankl,Austria,6,"9', 12', 18', 20', 53', and 66'",Austria,Malta,1977-04-30
7358,MATCH_12905,Steve Sumner,New Zealand,6,"8', 45', 55', 60', 72', and 86'",New Zealand,Fiji,1981-08-16
14134,MATCH_22111,Kazuyoshi Miura,Japan,6,"23', 29', 44', 57', 62', and 79'",Japan,Macau,1997-06-22
16850,MATCH_25421,John Aloisi,Australia,6,"14', 24', 37', 45', 52', and 63'",Australia,Tonga,2001-04-09
16522,MATCH_25082,Karim Bagheri,Iran,6,"13', 23', 33', 48', 67', and 77'",Iran,Guam,2000-11-24


## Cell 73 — Generate scoring-minute questions

In [ ]:
def create_scoring_minute_answer(row):
    scorer = row["scorer"]
    minutes = row["scoring_minutes"]

    return (
        f"{scorer} scored at {minutes}."
    )


player_goal_minute_qa_df = (
    player_match_minute_source.copy()
)

player_goal_minute_qa_df["question_id"] = [
    f"PGMINUTE_{number:05d}"
    for number in range(
        1,
        len(player_goal_minute_qa_df) + 1
    )
]

player_goal_minute_qa_df["intent"] = (
    "player_match_scoring_minutes"
)

player_goal_minute_qa_df["question"] = (
    "When did "
    + player_goal_minute_qa_df["scorer"]
    + " score in the match between "
    + player_goal_minute_qa_df[
        "home_team_standardized"
    ]
    + " and "
    + player_goal_minute_qa_df[
        "away_team_standardized"
    ]
    + " on "
    + player_goal_minute_qa_df[
        "date_standardized"
    ]
    + "?"
)

player_goal_minute_qa_df["answer"] = (
    player_goal_minute_qa_df.apply(
        create_scoring_minute_answer,
        axis=1
    )
)

player_goal_minute_qa_df = (
    player_goal_minute_qa_df[
        [
            "question_id",
            "match_id",
            "intent",
            "question",
            "answer",
            "scorer",
            "team",
            "goal_count",
            "scoring_minutes",
            "date_standardized",
            "home_team_standardized",
            "away_team_standardized",
            "home_score",
            "away_score",
            "tournament"
        ]
    ]
)

assert len(player_goal_minute_qa_df) == 39_758
assert player_goal_minute_qa_df["question_id"].is_unique
assert player_goal_minute_qa_df["question"].is_unique
assert player_goal_minute_qa_df["answer"].notna().all()

print(
    "Scoring-minute questions generated:",
    f"{len(player_goal_minute_qa_df):,}"
)

print("Scoring-minute question validation passed.")

display(
    player_goal_minute_qa_df[
        [
            "question_id",
            "question",
            "answer",
            "tournament"
        ]
    ].sample(
        n=10,
        random_state=42
    )
)

Scoring-minute questions generated: 39,758
Scoring-minute question validation passed.


,question_id,question,answer,tournament
8925,PGMINUTE_08926,When did Manuel Amoros score in the match between France and Belgium on 1986-06-28?,Manuel Amoros scored at 111'.,FIFA World Cup
4289,PGMINUTE_04290,When did Claude Barthélemy score in the match between El Salvador and Haiti on 1969-09-28?,Claude Barthélemy scored at 44'.,FIFA World Cup qualification
1213,PGMINUTE_01214,When did Víctor Ugarte score in the match between Bolivia and Uruguay on 1949-04-17?,Víctor Ugarte scored at 47'.,Copa América
740,PGMINUTE_00741,When did Franz Binder score in the match between Austria and Latvia on 1937-10-05?,Franz Binder scored at 33'.,FIFA World Cup qualification
3986,PGMINUTE_03987,When did Giora Spiegel score in the match between Israel and Taiwan on 1968-05-17?,Giora Spiegel scored at 76'.,AFC Asian Cup
3785,PGMINUTE_03786,When did Vojtech Masný score in the match between Republic of Ireland and Czechoslovakia on 1967-05-21?,Vojtech Masný scored at 47'.,UEFA Euro qualification
5488,PGMINUTE_05489,When did Gerd Müller score in the match between Germany and Netherlands on 1974-07-07?,Gerd Müller scored at 43'.,FIFA World Cup
29321,PGMINUTE_29322,When did Yousef Al-Naber score in the match between Jordan and Bangladesh on 2016-03-24?,Yousef Al-Naber scored at 82'.,FIFA World Cup qualification
39518,PGMINUTE_39519,When did Fisnik Asllani score in the match between Slovakia and Kosovo on 2026-03-26?,Fisnik Asllani scored at 47'.,FIFA World Cup qualification
17720,PGMINUTE_17721,When did Fan Zhiyi score in the match between China and Uzbekistan on 2001-09-15?,Fan Zhiyi scored at 76'.,FIFA World Cup qualification


## Cell 74 — Validate special scoring-minute cases

In [ ]:
scoring_minute_validation = pd.DataFrame([
    {
        "check": "Generated scoring-minute questions",
        "value": len(player_goal_minute_qa_df)
    },
    {
        "check": "Included goal events",
        "value": int(
            player_goal_minute_qa_df[
                "goal_count"
            ].sum()
        )
    },
    {
        "check": "Excluded incomplete player-match records",
        "value": int(
            (~player_minute_completeness[
                "all_minutes_known"
            ]).sum()
        )
    },
    {
        "check": "Excluded missing-minute events",
        "value": int(
            player_minute_completeness[
                "missing_minute_count"
            ].sum()
        )
    },
    {
        "check": "Duplicate question IDs",
        "value": int(
            player_goal_minute_qa_df[
                "question_id"
            ].duplicated().sum()
        )
    },
    {
        "check": "Duplicate question text",
        "value": int(
            player_goal_minute_qa_df[
                "question"
            ].duplicated().sum()
        )
    },
    {
        "check": "Missing answers",
        "value": int(
            player_goal_minute_qa_df[
                "answer"
            ].isna().sum()
        )
    }
])

display(scoring_minute_validation)

assert (
    player_goal_minute_qa_df["goal_count"].sum()
    == len(reliable_minute_events)
    == 46_655
)

assert len(player_goal_minute_qa_df) == 39_758

print("Scoring-minute dataset fully validated.")

,check,value
0,Generated scoring-minute questions,39758
1,Included goal events,46655
2,Excluded incomplete player-match records,173
3,Excluded missing-minute events,208
4,Duplicate question IDs,0
5,Duplicate question text,0
6,Missing answers,0


Scoring-minute dataset fully validated.


## Cell 75 — Inventory all generated QA DataFrames

In [ ]:
qa_dataframe_inventory = []

for variable_name, variable_value in list(globals().items()):
    if (
        variable_name.endswith("_qa_df")
        and isinstance(variable_value, pd.DataFrame)
    ):
        qa_dataframe_inventory.append({
            "dataframe": variable_name,
            "rows": len(variable_value),
            "columns": len(variable_value.columns),
            "has_question_id": (
                "question_id" in variable_value.columns
            ),
            "has_intent": (
                "intent" in variable_value.columns
            ),
            "has_question": (
                "question" in variable_value.columns
            ),
            "has_answer": (
                "answer" in variable_value.columns
            )
        })

qa_dataframe_inventory_df = (
    pd.DataFrame(qa_dataframe_inventory)
    .sort_values("dataframe")
    .reset_index(drop=True)
)

display(qa_dataframe_inventory_df)

print("\nColumn names for each QA DataFrame:\n")

for dataframe_name in qa_dataframe_inventory_df["dataframe"]:
    dataframe_value = globals()[dataframe_name]

    print(f"{dataframe_name}:")
    print(dataframe_value.columns.tolist())
    print()

,dataframe,rows,columns,has_question_id,has_intent,has_question,has_answer
0,player_goal_count_qa_df,39931,14,True,True,True,True
1,player_goal_minute_qa_df,39758,15,True,True,True,True
2,result_based_qa_df,98962,11,True,True,True,True
3,score_qa_df,49481,11,True,True,True,True
4,scorer_list_qa_df,15508,11,True,True,True,True
5,winner_qa_df,49481,11,True,True,True,True



Column names for each QA DataFrame:

player_goal_count_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'scorer', 'team', 'player_goal_count', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

player_goal_minute_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'scorer', 'team', 'goal_count', 'scoring_minutes', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

result_based_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

score_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

scorer_list_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'date

## 8. Final QA Dataset Construction

The validated question datasets are now combined into one final football question-answering dataset.

The combined dataset contains four question groups:

- Match winner and match score questions from `result_based_qa_df`
- Match scorer-list questions from `scorer_list_qa_df`
- Player goal-count questions from `player_goal_count_qa_df`
- Player scoring-minute questions from `player_goal_minute_qa_df`

The separate `winner_qa_df` and `score_qa_df` DataFrames are not added again because they are already contained in `result_based_qa_df`.

## Cell 77 — Inspect intent values before combining

In [ ]:
qa_sources = {
    "result_based": result_based_qa_df,
    "scorer_list": scorer_list_qa_df,
    "player_goal_count": player_goal_count_qa_df,
    "player_goal_minute": player_goal_minute_qa_df
}

intent_inventory = []

for source_name, source_df in qa_sources.items():
    intent_counts = (
        source_df["intent"]
        .value_counts(dropna=False)
    )

    for intent_name, row_count in intent_counts.items():
        intent_inventory.append({
            "source": source_name,
            "intent": intent_name,
            "rows": int(row_count)
        })

intent_inventory_df = (
    pd.DataFrame(intent_inventory)
    .sort_values(
        ["source", "intent"]
    )
    .reset_index(drop=True)
)

display(intent_inventory_df)

print(
    "Expected rows after combination:",
    f"{sum(len(df) for df in qa_sources.values()):,}"
)

,source,intent,rows
0,player_goal_count,player_match_goal_count,39931
1,player_goal_minute,player_match_scoring_minutes,39758
2,result_based,match_score,49481
3,result_based,match_winner,49481
4,scorer_list,match_scorers,15508


Expected rows after combination: 194,159


## Cell 78 — Combine the validated QA datasets

In [ ]:
final_qa_parts = []

for source_name, source_df in qa_sources.items():
    prepared_df = source_df.copy()
    prepared_df["source_dataset"] = source_name
    final_qa_parts.append(prepared_df)

final_qa_df = pd.concat(
    final_qa_parts,
    ignore_index=True,
    sort=False
)

preferred_column_order = [
    "question_id",
    "match_id",
    "intent",
    "question",
    "answer",
    "source_dataset",
    "scorer",
    "team",
    "player_goal_count",
    "goal_count",
    "scoring_minutes",
    "date_standardized",
    "home_team_standardized",
    "away_team_standardized",
    "home_score",
    "away_score",
    "tournament"
]

final_qa_df = final_qa_df[
    preferred_column_order
]

print(
    "Final QA dataset rows:",
    f"{len(final_qa_df):,}"
)

print(
    "Final QA dataset columns:",
    len(final_qa_df.columns)
)

display(final_qa_df.head())

Final QA dataset rows: 194,159
Final QA dataset columns: 17


,question_id,match_id,intent,question,answer,source_dataset,scorer,team,player_goal_count,goal_count,scoring_minutes,date_standardized,home_team_standardized,away_team_standardized,home_score,away_score,tournament
0,WIN_00001,MATCH_00001,match_winner,Who won the match between Scotland and England on 1872-11-30?,The match between Scotland and England ended in a 0-0 draw on 1872-11-30.,result_based,NaN,NaN,NaN,NaN,NaN,1872-11-30,Scotland,England,0,0,Friendly
1,WIN_00002,MATCH_00002,match_winner,Who won the match between England and Scotland on 1873-03-08?,England won against Scotland 4-2 on 1873-03-08.,result_based,NaN,NaN,NaN,NaN,NaN,1873-03-08,England,Scotland,4,2,Friendly
2,WIN_00003,MATCH_00003,match_winner,Who won the match between Scotland and England on 1874-03-07?,Scotland won against England 2-1 on 1874-03-07.,result_based,NaN,NaN,NaN,NaN,NaN,1874-03-07,Scotland,England,2,1,Friendly
3,WIN_00004,MATCH_00004,match_winner,Who won the match between England and Scotland on 1875-03-06?,The match between England and Scotland ended in a 2-2 draw on 1875-03-06.,result_based,NaN,NaN,NaN,NaN,NaN,1875-03-06,England,Scotland,2,2,Friendly
4,WIN_00005,MATCH_00005,match_winner,Who won the match between Scotland and England on 1876-03-04?,Scotland won against England 3-0 on 1876-03-04.,result_based,NaN,NaN,NaN,NaN,NaN,1876-03-04,Scotland,England,3,0,Friendly


## Cell 79 — Validate the final dataset structure

In [ ]:
final_dataset_validation = pd.DataFrame([
    {
        "check": "Expected combined rows",
        "value": sum(
            len(df)
            for df in qa_sources.values()
        )
    },
    {
        "check": "Actual combined rows",
        "value": len(final_qa_df)
    },
    {
        "check": "Duplicate question IDs",
        "value": int(
            final_qa_df[
                "question_id"
            ].duplicated().sum()
        )
    },
    {
        "check": "Duplicate question text",
        "value": int(
            final_qa_df[
                "question"
            ].duplicated().sum()
        )
    },
    {
        "check": "Missing question IDs",
        "value": int(
            final_qa_df[
                "question_id"
            ].isna().sum()
        )
    },
    {
        "check": "Missing match IDs",
        "value": int(
            final_qa_df[
                "match_id"
            ].isna().sum()
        )
    },
    {
        "check": "Missing intents",
        "value": int(
            final_qa_df[
                "intent"
            ].isna().sum()
        )
    },
    {
        "check": "Missing questions",
        "value": int(
            final_qa_df[
                "question"
            ].isna().sum()
        )
    },
    {
        "check": "Missing answers",
        "value": int(
            final_qa_df[
                "answer"
            ].isna().sum()
        )
    }
])

display(final_dataset_validation)

assert len(final_qa_df) == 194_159
assert final_qa_df["question_id"].is_unique
assert final_qa_df["question"].is_unique

assert final_qa_df[
    [
        "question_id",
        "match_id",
        "intent",
        "question",
        "answer"
    ]
].notna().all().all()

print("Final QA dataset structural validation passed.")

,check,value
0,Expected combined rows,194159
1,Actual combined rows,194159
2,Duplicate question IDs,0
3,Duplicate question text,0
4,Missing question IDs,0
5,Missing match IDs,0
6,Missing intents,0
7,Missing questions,0
8,Missing answers,0


Final QA dataset structural validation passed.


## Cell 80 — Validate row totals by source

In [ ]:
source_count_validation = (
    final_qa_df[
        "source_dataset"
    ]
    .value_counts()
    .rename_axis("source_dataset")
    .reset_index(name="actual_rows")
)

expected_source_counts = pd.DataFrame([
    {
        "source_dataset": source_name,
        "expected_rows": len(source_df)
    }
    for source_name, source_df in qa_sources.items()
])

source_count_validation = (
    expected_source_counts
    .merge(
        source_count_validation,
        on="source_dataset",
        how="left",
        validate="one_to_one"
    )
)

source_count_validation["difference"] = (
    source_count_validation["actual_rows"]
    - source_count_validation["expected_rows"]
)

display(source_count_validation)

assert source_count_validation["difference"].eq(0).all()
assert source_count_validation["actual_rows"].sum() == 194_159

print("All source dataset row totals preserved.")

,source_dataset,expected_rows,actual_rows,difference
0,result_based,98962,98962,0
1,scorer_list,15508,15508,0
2,player_goal_count,39931,39931,0
3,player_goal_minute,39758,39758,0


All source dataset row totals preserved.


## Cell 81 — Inspect final intent distribution

In [ ]:
final_intent_summary = (
    final_qa_df
    .groupby(
        ["intent", "source_dataset"],
        dropna=False
    )
    .size()
    .reset_index(name="question_count")
    .sort_values(
        "question_count",
        ascending=False
    )
    .reset_index(drop=True)
)

display(final_intent_summary)

print(
    "Number of unique intents:",
    final_qa_df["intent"].nunique()
)

print(
    "Total questions:",
    f"{final_intent_summary['question_count'].sum():,}"
)

,intent,source_dataset,question_count
0,match_score,result_based,49481
1,match_winner,result_based,49481
2,player_match_goal_count,player_goal_count,39931
3,player_match_scoring_minutes,player_goal_minute,39758
4,match_scorers,scorer_list,15508


Number of unique intents: 5
Total questions: 194,159


## Cell 82 — Inspect random final questions

In [ ]:
final_sample = (
    final_qa_df[
        [
            "question_id",
            "intent",
            "question",
            "answer",
            "source_dataset"
        ]
    ]
    .sample(
        n=20,
        random_state=42
    )
)

display(final_sample)

,question_id,intent,question,answer,source_dataset
20383,WIN_20384,match_winner,Who won the match between Costa Rica and Ecuador on 1995-06-10?,Ecuador won against Costa Rica 2-1 on 1995-06-10.,result_based
187676,PGMINUTE_33276,player_match_scoring_minutes,When did Mathias Jensen score in the match between Denmark and Moldova on 2021-03-28?,Mathias Jensen scored at 39'.,player_goal_minute
39568,WIN_39569,match_winner,Who won the match between Seychelles and Lesotho on 2016-03-26?,Seychelles won against Lesotho 2-0 on 2016-03-26.,result_based
98196,SCORE_48716,match_score,What was the score between Timor-Leste and Philippines on 2025-10-09?,Timor-Leste 1-4 Philippines.,result_based
162318,PGMINUTE_07918,player_match_scoring_minutes,When did Ruud Gullit score in the match between Netherlands and Iceland on 1983-09-07?,Ruud Gullit scored at 19'.,player_goal_minute
132926,PGCOUNT_18457,player_match_goal_count,How many goals did Miroslav Klose score in the match between Germany and Faroe Islands on 2002-10-16?,Miroslav Klose scored 1 goal for Germany in the match.,player_goal_count
137539,PGCOUNT_23070,player_match_goal_count,How many goals did David Villa score in the match between Sweden and Spain on 2008-06-14?,David Villa scored 1 goal for Spain in the match.,player_goal_count
190707,PGMINUTE_36307,player_match_scoring_minutes,When did Darwin Núñez score in the match between Argentina and Uruguay on 2023-11-16?,Darwin Núñez scored at 87'.,player_goal_minute
150804,PGCOUNT_36335,player_match_goal_count,How many goals did Niki Torrão score in the match between Myanmar and Macau on 2023-10-12?,Niki Torrão scored 1 goal for Macau in the match.,player_goal_count
27638,WIN_27639,match_winner,Who won the match between Tanzania and Kenya on 2003-10-11?,The match between Tanzania and Kenya ended in a 0-0 draw on 2003-10-11.,result_based


## 8.1 Final Quality Assurance

Before export, the combined dataset is checked for:

- Representation of every intent
- Correct source-to-intent mapping
- Agreement between answers and structured match data
- Valid player-team relationships
- Valid score values
- Missing or blank text
- Unintended whitespace
- Duplicate questions and identifiers

Stratified sampling is used so that every question type is inspected, including less frequent intents.

## Cell 84 — Inspect samples from every intent

In [ ]:
intent_samples = (
    final_qa_df
    .groupby(
        "intent",
        group_keys=False
    )
    .sample(
        n=5,
        random_state=42
    )
    [
        [
            "question_id",
            "intent",
            "question",
            "answer",
            "source_dataset"
        ]
    ]
    .sort_values(
        [
            "intent",
            "question_id"
        ]
    )
)

display(intent_samples)

assert (
    intent_samples["intent"].value_counts()
    .eq(5)
    .all()
)

print("Five examples from every intent displayed.")

,question_id,intent,question,answer,source_dataset
67464,SCORE_17984,match_score,What was the score between Mali and Tunisia on 1991-09-26?,Mali 0-0 Tunisia.,result_based
70516,SCORE_21036,match_score,What was the score between Jamaica and Saint Kitts and Nevis on 1996-05-26?,Jamaica 4-1 Saint Kitts and Nevis.,result_based
72099,SCORE_22619,match_score,What was the score between Cameroon and Algeria on 1998-02-15?,Cameroon 2-1 Algeria.,result_based
84647,SCORE_35167,match_score,What was the score between Zimbabwe and Liberia on 2011-09-04?,Zimbabwe 3-0 Liberia.,result_based
90056,SCORE_40576,match_score,What was the score between Burkina Faso and Benin on 2017-05-04?,Burkina Faso 1-1 Benin.,result_based
101605,SCORERS_02644,match_scorers,Who scored in the match between Nigeria and Guinea on 1981-04-15?,The recorded scorers were Henry Nwosu for Nigeria.,scorer_list
109523,SCORERS_10562,match_scorers,Who scored in the match between Gambia and Tanzania on 2013-09-07?,The recorded scorers were Mustapha Jarju for Gambia and Mustapha Jarju for Gambia.,scorer_list
109894,SCORERS_10933,match_scorers,Who scored in the match between Bulgaria and Italy on 2015-03-28?,"The recorded scorers were Yordan Minev (own goal credited to Italy), Ivelin Popov for Bulgaria, Iliyan Mitsanski for Bulgaria, and Éder for Italy.",scorer_list
112529,SCORERS_13568,match_scorers,Who scored in the match between Mozambique and Malawi on 2021-11-16?,The recorded scorers were Limbikani Mzava (own goal credited to Mozambique).,scorer_list
113376,SCORERS_14415,match_scorers,Who scored in the match between Qatar and Kuwait on 2024-03-21?,"The recorded scorers were Akram Afif for Qatar, Ahmed Al-Rawi for Qatar, and Akram Afif for Qatar.",scorer_list


Five examples from every intent displayed.


## Cell 85 — Validate source and intent relationships

In [ ]:
expected_intent_sources = {
    "match_winner": "result_based",
    "match_score": "result_based",
    "match_scorers": "scorer_list",
    "player_match_goal_count": "player_goal_count",
    "player_match_scoring_minutes": "player_goal_minute"
}

actual_intent_sources = (
    final_qa_df[
        [
            "intent",
            "source_dataset"
        ]
    ]
    .drop_duplicates()
    .sort_values("intent")
    .reset_index(drop=True)
)

actual_intent_sources["expected_source"] = (
    actual_intent_sources["intent"]
    .map(expected_intent_sources)
)

actual_intent_sources["source_matches"] = (
    actual_intent_sources["source_dataset"]
    .eq(
        actual_intent_sources[
            "expected_source"
        ]
    )
)

display(actual_intent_sources)

assert actual_intent_sources["source_matches"].all()
assert set(final_qa_df["intent"]) == set(
    expected_intent_sources
)

print("All intents are connected to the correct source datasets.")

,intent,source_dataset,expected_source,source_matches
0,match_score,result_based,result_based,True
1,match_scorers,scorer_list,scorer_list,True
2,match_winner,result_based,result_based,True
3,player_match_goal_count,player_goal_count,player_goal_count,True
4,player_match_scoring_minutes,player_goal_minute,player_goal_minute,True


All intents are connected to the correct source datasets.


## Cell 86 — Validate structured values

In [ ]:
player_rows = final_qa_df[
    final_qa_df["intent"].isin([
        "player_match_goal_count",
        "player_match_scoring_minutes"
    ])
].copy()

structured_value_validation = pd.DataFrame([
    {
        "check": "Negative home scores",
        "value": int(
            final_qa_df["home_score"].lt(0).sum()
        )
    },
    {
        "check": "Negative away scores",
        "value": int(
            final_qa_df["away_score"].lt(0).sum()
        )
    },
    {
        "check": "Player rows with missing scorer",
        "value": int(
            player_rows["scorer"].isna().sum()
        )
    },
    {
        "check": "Player rows with missing team",
        "value": int(
            player_rows["team"].isna().sum()
        )
    },
    {
        "check": "Player teams not participating in match",
        "value": int(
            (
                ~player_rows["team"].eq(
                    player_rows[
                        "home_team_standardized"
                    ]
                )
                & ~player_rows["team"].eq(
                    player_rows[
                        "away_team_standardized"
                    ]
                )
            ).sum()
        )
    },
    {
        "check": "Non-positive player goal counts",
        "value": int(
            final_qa_df.loc[
                final_qa_df["intent"].eq(
                    "player_match_goal_count"
                ),
                "player_goal_count"
            ].le(0).sum()
        )
    },
    {
        "check": "Non-positive scoring-minute goal counts",
        "value": int(
            final_qa_df.loc[
                final_qa_df["intent"].eq(
                    "player_match_scoring_minutes"
                ),
                "goal_count"
            ].le(0).sum()
        )
    }
])

display(structured_value_validation)

assert structured_value_validation["value"].eq(0).all()

print("Structured-value validation passed.")

,check,value
0,Negative home scores,0
1,Negative away scores,0
2,Player rows with missing scorer,0
3,Player rows with missing team,0
4,Player teams not participating in match,0
5,Non-positive player goal counts,0
6,Non-positive scoring-minute goal counts,0


Structured-value validation passed.


## Cell 87 — Verify generated answers against structured data

In [ ]:
def expected_winner_answer(row):
    home_team = row["home_team_standardized"]
    away_team = row["away_team_standardized"]
    home_score = int(row["home_score"])
    away_score = int(row["away_score"])
    match_date = row["date_standardized"]

    if home_score > away_score:
        return (
            f"{home_team} won against {away_team} "
            f"{home_score}-{away_score} on {match_date}."
        )

    if away_score > home_score:
        return (
            f"{away_team} won against {home_team} "
            f"{away_score}-{home_score} on {match_date}."
        )

    return (
        f"The match between {home_team} and {away_team} "
        f"ended in a {home_score}-{away_score} draw "
        f"on {match_date}."
    )


def expected_score_answer(row):
    return (
        f"{row['home_team_standardized']} "
        f"{int(row['home_score'])}-"
        f"{int(row['away_score'])} "
        f"{row['away_team_standardized']}."
    )


def expected_goal_count_answer(row):
    goal_count = int(row["player_goal_count"])
    goal_word = "goal" if goal_count == 1 else "goals"

    return (
        f"{row['scorer']} scored {goal_count} "
        f"{goal_word} for {row['team']} in the match."
    )


def expected_minute_answer(row):
    return (
        f"{row['scorer']} scored at "
        f"{row['scoring_minutes']}."
    )


answer_builders = {
    "match_winner": expected_winner_answer,
    "match_score": expected_score_answer,
    "player_match_goal_count": expected_goal_count_answer,
    "player_match_scoring_minutes": expected_minute_answer
}

answer_validation_rows = []

for intent_name, answer_builder in answer_builders.items():
    intent_df = final_qa_df[
        final_qa_df["intent"].eq(intent_name)
    ].copy()

    expected_answers = intent_df.apply(
        answer_builder,
        axis=1
    )

    mismatch_count = int(
        (~intent_df["answer"].eq(expected_answers)).sum()
    )

    answer_validation_rows.append({
        "intent": intent_name,
        "checked_rows": len(intent_df),
        "answer_mismatches": mismatch_count
    })

answer_validation_df = pd.DataFrame(
    answer_validation_rows
)

display(answer_validation_df)

assert answer_validation_df[
    "answer_mismatches"
].eq(0).all()

print("All reconstructable answers match their structured data.")

,intent,checked_rows,answer_mismatches
0,match_winner,49481,0
1,match_score,49481,0
2,player_match_goal_count,39931,0
3,player_match_scoring_minutes,39758,0


All reconstructable answers match their structured data.


## Cell 88 — Check text cleanliness

In [ ]:
text_columns = [
    "question_id",
    "match_id",
    "intent",
    "question",
    "answer",
    "source_dataset"
]

text_quality_rows = []

for column_name in text_columns:
    text_values = (
        final_qa_df[column_name]
        .astype("string")
    )

    text_quality_rows.append({
        "column": column_name,
        "missing_values": int(
            final_qa_df[column_name]
            .isna()
            .sum()
        ),
        "blank_values": int(
            text_values.str.strip()
            .eq("")
            .sum()
        ),
        "leading_or_trailing_spaces": int(
            text_values.ne(
                text_values.str.strip()
            ).sum()
        ),
        "repeated_spaces": int(
            text_values.str.contains(
                r" {2,}",
                regex=True,
                na=False
            ).sum()
        )
    })

text_quality_summary = pd.DataFrame(
    text_quality_rows
)

display(text_quality_summary)

assert (
    text_quality_summary[
        [
            "missing_values",
            "blank_values",
            "leading_or_trailing_spaces",
            "repeated_spaces"
        ]
    ].eq(0).all().all()
)

print("Final text-quality validation passed.")

,column,missing_values,blank_values,leading_or_trailing_spaces,repeated_spaces
0,question_id,0,0,0,0
1,match_id,0,0,0,0
2,intent,0,0,0,0
3,question,0,0,0,0
4,answer,0,0,0,0
5,source_dataset,0,0,0,0


Final text-quality validation passed.


## Cell 89 — Inventory possible scorer source DataFrames

In [ ]:
scorer_source_inventory = []

scorer_related_terms = [
    "scorer",
    "team",
    "own_goal",
    "match_id"
]

for variable_name, variable_value in list(globals().items()):
    if not isinstance(variable_value, pd.DataFrame):
        continue

    column_names = set(variable_value.columns)

    relevance_score = sum(
        term in column_names
        for term in scorer_related_terms
    )

    if (
        relevance_score >= 2
        or "scorer" in variable_name.lower()
        or "goal" in variable_name.lower()
    ):
        scorer_source_inventory.append({
            "dataframe": variable_name,
            "rows": len(variable_value),
            "columns": len(variable_value.columns),
            "has_match_id": "match_id" in column_names,
            "has_scorer": "scorer" in column_names,
            "has_team": "team" in column_names,
            "has_own_goal": "own_goal" in column_names,
            "has_minute": "minute" in column_names
        })

scorer_source_inventory_df = (
    pd.DataFrame(scorer_source_inventory)
    .sort_values(
        ["has_scorer", "has_match_id", "rows"],
        ascending=[False, False, False]
    )
    .reset_index(drop=True)
)

display(scorer_source_inventory_df)

print("\nColumns in likely scorer source DataFrames:\n")

for dataframe_name in scorer_source_inventory_df.loc[
    scorer_source_inventory_df["has_scorer"],
    "dataframe"
]:
    print(f"{dataframe_name}:")
    print(globals()[dataframe_name].columns.tolist())
    print()

,dataframe,rows,columns,has_match_id,has_scorer,has_team,has_own_goal,has_minute
0,final_qa_df,194159,17,True,True,True,False,False
1,player_rows,79689,17,True,True,True,False,False
2,goalscorer_question_source,47839,20,True,True,True,True,True
3,reliable_scorer_events,47791,21,True,True,True,True,True
4,player_goal_events,46865,21,True,True,True,True,True
5,reliable_minute_events,46655,23,True,True,True,True,True
6,player_match_goal_source,39931,11,True,True,True,False,False
7,player_goal_count_qa_df,39931,14,True,True,True,False,False
8,player_minute_completeness,39931,7,True,True,True,False,False
9,reliable_minute_groups,39758,3,True,True,True,False,False



Columns in likely scorer source DataFrames:

final_qa_df:
['question_id', 'match_id', 'intent', 'question', 'answer', 'source_dataset', 'scorer', 'team', 'player_goal_count', 'goal_count', 'scoring_minutes', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

player_rows:
['question_id', 'match_id', 'intent', 'question', 'answer', 'source_dataset', 'scorer', 'team', 'player_goal_count', 'goal_count', 'scoring_minutes', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'home_score', 'away_score', 'tournament']

goalscorer_question_source:
['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty', 'date_parsed', 'home_team_key', 'away_team_key', 'safe_result_key', 'date_standardized', 'home_team_standardized', 'away_team_standardized', 'minute_original', 'base_minute', 'added_time', 'minute_total', 'match_id']

reliable_scorer_events:
['date', 'home_team', 'away_team', 'team

## Cell 90 — Measure repeated-name scorer answers

In [ ]:
repeated_scorer_examples = (
    scorer_list_qa_df[
        scorer_list_qa_df["answer"].str.contains(
            r"\b(.+?) for .+ and \1 for ",
            regex=True,
            na=False
        )
    ][
        [
            "question_id",
            "match_id",
            "question",
            "answer"
        ]
    ]
)

print(
    "Potential answers containing repeated scorer names:",
    f"{len(repeated_scorer_examples):,}"
)

display(repeated_scorer_examples.head(20))

Potential answers containing repeated scorer names: 2,551


,question_id,match_id,question,answer
0,SCORERS_00001,MATCH_00480,Who scored in the match between Chile and Uruguay on 1916-07-02?,"The recorded scorers were José Piendibene for Uruguay, Isabelino Gradín for Uruguay, Isabelino Gradín for Uruguay, and José Piendibene for Uruguay."
1,SCORERS_00002,MATCH_00482,Who scored in the match between Argentina and Chile on 1916-07-06?,"The recorded scorers were Alberto Ohaco for Argentina, Telésforo Báez for Chile, Juan Domingo Brown for Argentina (penalty), Juan Domingo Brown fo..."
5,SCORERS_00006,MATCH_00516,Who scored in the match between Uruguay and Chile on 1917-09-30?,"The recorded scorers were Carlos Scarone for Uruguay, Ángel Romano for Uruguay, Carlos Scarone for Uruguay (penalty), and Ángel Romano for Uruguay."
9,SCORERS_00010,MATCH_00522,Who scored in the match between Brazil and Chile on 1917-10-12?,"The recorded scorers were Caetano Izzo for Brazil, Neco for Brazil, Haroldo Domingues for Brazil, Amílcar Barbuy for Brazil, and Haroldo Domingues..."
11,SCORERS_00012,MATCH_00549,Who scored in the match between Brazil and Chile on 1919-05-11?,"The recorded scorers were Arthur Friedenreich for Brazil, Neco for Brazil, Arthur Friedenreich for Brazil, Arthur Friedenreich for Brazil, Haroldo..."
15,SCORERS_00016,MATCH_00559,Who scored in the match between Argentina and Chile on 1919-05-22?,"The recorded scorers were Edwin Clarcke for Argentina, Carlos Izaguirre for Argentina, Edwin Clarcke for Argentina, Alfredo France for Chile, and ..."
16,SCORERS_00017,MATCH_00561,Who scored in the match between Brazil and Uruguay on 1919-05-26?,"The recorded scorers were Isabelino Gradín for Uruguay, Carlos Scarone for Uruguay, Neco for Brazil, and Neco for Brazil."
20,SCORERS_00021,MATCH_00638,Who scored in the match between Brazil and Uruguay on 1920-09-18?,"The recorded scorers were Ángel Romano for Uruguay, Antonio Urdinarán for Uruguay (penalty), José Pérez for Uruguay, Antonio Campolo for Uruguay, ..."
38,SCORERS_00039,MATCH_00769,Who scored in the match between Argentina and Paraguay on 1922-10-18?,The recorded scorers were Julio Francia for Argentina and Julio Francia for Argentina (penalty).
39,SCORERS_00040,MATCH_00774,Who scored in the match between Brazil and Paraguay on 1922-11-06?,"The recorded scorers were Neco for Brazil, Xavier Camargo for Brazil, and Xavier Camargo for Brazil."


## Cell 91 — Rebuild aggregated scorer-list answers

In [ ]:
def normalize_boolean(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
        .fillna(False)
        .astype(bool)
    )


scorer_answer_events = reliable_scorer_events[
    [
        "match_id",
        "scorer",
        "team",
        "own_goal",
        "penalty",
        "minute_total"
    ]
].copy()

scorer_answer_events["own_goal"] = normalize_boolean(
    scorer_answer_events["own_goal"]
)

scorer_answer_events["penalty"] = normalize_boolean(
    scorer_answer_events["penalty"]
)

# Preserve the source order as a stable fallback,
# especially for events without recorded minutes.
scorer_answer_events["source_order"] = range(
    len(scorer_answer_events)
)

scorer_answer_events = scorer_answer_events.sort_values(
    [
        "match_id",
        "minute_total",
        "source_order"
    ],
    kind="stable",
    na_position="last"
).reset_index(drop=True)

scorer_answer_events["event_order"] = (
    scorer_answer_events
    .groupby("match_id")
    .cumcount()
)

aggregated_scorer_events = (
    scorer_answer_events
    .groupby(
        [
            "match_id",
            "scorer",
            "team",
            "own_goal"
        ],
        as_index=False,
        sort=False,
        dropna=False
    )
    .agg(
        goal_count=("scorer", "size"),
        penalty_count=("penalty", "sum"),
        first_event_order=("event_order", "min")
    )
    .sort_values(
        [
            "match_id",
            "first_event_order"
        ],
        kind="stable"
    )
    .reset_index(drop=True)
)


def format_scorer_summary(row):
    scorer = row["scorer"]
    credited_team = row["team"]
    goal_count = int(row["goal_count"])
    penalty_count = int(row["penalty_count"])

    if row["own_goal"]:
        if goal_count == 1:
            return (
                f"{scorer} scored an own goal "
                f"credited to {credited_team}"
            )

        return (
            f"{scorer} scored {goal_count} own goals "
            f"credited to {credited_team}"
        )

    goal_word = "goal" if goal_count == 1 else "goals"

    summary = (
        f"{scorer} scored {goal_count} "
        f"{goal_word} for {credited_team}"
    )

    if penalty_count == 1:
        summary += ", including 1 penalty"
    elif penalty_count > 1:
        summary += f", including {penalty_count} penalties"

    return summary


aggregated_scorer_events["scorer_summary"] = (
    aggregated_scorer_events.apply(
        format_scorer_summary,
        axis=1
    )
)

display(
    aggregated_scorer_events.head(20)
)

,match_id,scorer,team,own_goal,goal_count,penalty_count,first_event_order,scorer_summary
0,MATCH_00480,José Piendibene,Uruguay,False,2,0,0,José Piendibene scored 2 goals for Uruguay
1,MATCH_00480,Isabelino Gradín,Uruguay,False,2,0,1,Isabelino Gradín scored 2 goals for Uruguay
2,MATCH_00482,Alberto Ohaco,Argentina,False,2,0,0,Alberto Ohaco scored 2 goals for Argentina
3,MATCH_00482,Telésforo Báez,Chile,False,1,0,1,Telésforo Báez scored 1 goal for Chile
4,MATCH_00482,Juan Domingo Brown,Argentina,False,2,2,2,"Juan Domingo Brown scored 2 goals for Argentina, including 2 penalties"
5,MATCH_00482,Alberto Marcovecchio,Argentina,False,2,0,4,Alberto Marcovecchio scored 2 goals for Argentina
6,MATCH_00483,Demóstenes Correia de Syllos,Brazil,False,1,0,0,Demóstenes Correia de Syllos scored 1 goal for Brazil
7,MATCH_00483,Hernando Salazar,Chile,False,1,0,1,Hernando Salazar scored 1 goal for Chile
8,MATCH_00484,José Durand Laguna,Argentina,False,1,0,0,José Durand Laguna scored 1 goal for Argentina
9,MATCH_00484,Manoel Alencar Monte,Brazil,False,1,0,1,Manoel Alencar Monte scored 1 goal for Brazil


## Cell 92 — Construct one natural answer per match

In [ ]:
def join_scorer_summaries(summary_values):
    summaries = list(summary_values)

    if len(summaries) == 1:
        joined_text = summaries[0]
    elif len(summaries) == 2:
        joined_text = (
            f"{summaries[0]} and {summaries[1]}"
        )
    else:
        joined_text = (
            ", ".join(summaries[:-1])
            + f", and {summaries[-1]}"
        )

    return f"{joined_text}."


rebuilt_scorer_answers = (
    aggregated_scorer_events
    .groupby(
        "match_id",
        sort=False
    )["scorer_summary"]
    .apply(join_scorer_summaries)
    .rename("rebuilt_answer")
    .reset_index()
)

print(
    "Rebuilt match-scorer answers:",
    f"{len(rebuilt_scorer_answers):,}"
)

display(
    rebuilt_scorer_answers.head(10)
)

assert len(rebuilt_scorer_answers) == len(
    scorer_list_qa_df
)

assert rebuilt_scorer_answers[
    "match_id"
].is_unique

print("One aggregated answer created for every scorer-list question.")

Rebuilt match-scorer answers: 15,508


,match_id,rebuilt_answer
0,MATCH_00480,José Piendibene scored 2 goals for Uruguay and Isabelino Gradín scored 2 goals for Uruguay.
1,MATCH_00482,"Alberto Ohaco scored 2 goals for Argentina, Telésforo Báez scored 1 goal for Chile, Juan Domingo Brown scored 2 goals for Argentina, including 2 p..."
2,MATCH_00483,Demóstenes Correia de Syllos scored 1 goal for Brazil and Hernando Salazar scored 1 goal for Chile.
3,MATCH_00484,José Durand Laguna scored 1 goal for Argentina and Manoel Alencar Monte scored 1 goal for Brazil.
4,MATCH_00486,"Arthur Friedenreich scored 1 goal for Brazil, Isabelino Gradín scored 1 goal for Uruguay, and Jose Tognola scored 1 goal for Uruguay."
5,MATCH_00516,"Carlos Scarone scored 2 goals for Uruguay, including 1 penalty and Ángel Romano scored 2 goals for Uruguay."
6,MATCH_00517,"Neco scored 1 goal for Brazil, Pedro Calomino scored 1 goal for Argentina, Silvio Lagreca scored 1 goal for Brazil, including 1 penalty, Alberto O..."
7,MATCH_00518,Luis García scored an own goal credited to Argentina.
8,MATCH_00521,"Héctor Scarone scored 1 goal for Uruguay, Ángel Romano scored 2 goals for Uruguay, and Carlos Scarone scored 1 goal for Uruguay."
9,MATCH_00522,"Caetano Izzo scored 1 goal for Brazil, Neco scored 1 goal for Brazil, Haroldo Domingues scored 2 goals for Brazil, and Amílcar Barbuy scored 1 goa..."


One aggregated answer created for every scorer-list question.


## Cell 93 — Update the scorer-list QA dataset

In [ ]:
scorer_list_qa_df = (
    scorer_list_qa_df
    .drop(columns=["answer"])
    .merge(
        rebuilt_scorer_answers.rename(
            columns={
                "rebuilt_answer": "answer"
            }
        ),
        on="match_id",
        how="left",
        validate="one_to_one"
    )
)

assert scorer_list_qa_df["answer"].notna().all()
assert scorer_list_qa_df["question_id"].is_unique
assert scorer_list_qa_df["question"].is_unique

print(
    "Updated scorer-list questions:",
    f"{len(scorer_list_qa_df):,}"
)

display(
    scorer_list_qa_df[
        [
            "question_id",
            "match_id",
            "question",
            "answer"
        ]
    ].sample(
        n=20,
        random_state=42
    )
)

Updated scorer-list questions: 15,508


,question_id,match_id,question,answer
6332,SCORERS_06333,MATCH_25167,Who scored in the match between Togo and Cameroon on 2001-01-28?,Samuel Eto'o scored 1 goal for Cameroon and Patrick M'Boma scored 1 goal for Cameroon.
2919,SCORERS_02920,MATCH_13862,Who scored in the match between Iceland and Republic of Ireland on 1983-09-21?,"Gary Waddock scored 1 goal for Republic of Ireland, Michael Robinson scored 1 goal for Republic of Ireland, and Mickey Walsh scored 1 goal for Rep..."
6165,SCORERS_06166,MATCH_24683,Who scored in the match between Venezuela and Bolivia on 2000-06-28?,"Miguel Mea Vitali scored 1 goal for Venezuela, Ruberth Morán scored 1 goal for Venezuela, Jaime Moreno scored 1 goal for Bolivia, Julio César Bald..."
3589,SCORERS_03590,MATCH_16585,Who scored in the match between Yemen and Syria on 1989-03-10?,Nizar Mahrous scored 1 goal for Syria.
10021,SCORERS_10022,MATCH_35615,Who scored in the match between Niger and Morocco on 2012-01-31?,Younès Belhanda scored 1 goal for Morocco.
6350,SCORERS_06351,MATCH_25225,Who scored in the match between Kuwait and Singapore on 2001-02-21?,Khalaf Al-Mutairi scored 1 goal for Kuwait.
14529,SCORERS_14530,MATCH_47344,Who scored in the match between Kuwait and Afghanistan on 2024-06-11?,Eid Al Rashidi scored 1 goal for Kuwait.
1768,SCORERS_01769,MATCH_09187,Who scored in the match between New Zealand and Tahiti on 1973-02-18?,Alan Vest scored 1 goal for New Zealand and Erroll Bennett scored 1 goal for Tahiti.
12334,SCORERS_12335,MATCH_42501,Who scored in the match between Ukraine and Serbia on 2019-06-07?,"Viktor Tsyhankov scored 2 goals for Ukraine, Yevhen Konoplyanka scored 2 goals for Ukraine, and Roman Yaremchuk scored 1 goal for Ukraine."
7934,SCORERS_07935,MATCH_29314,Who scored in the match between United States and Cuba on 2005-07-07?,"Lester Moré scored 1 goal for Cuba, Clint Dempsey scored 1 goal for United States, Landon Donovan scored 2 goals for United States, and DaMarcus B..."


## Cell 94 — Recombine the final dataset

In [ ]:
qa_sources = {
    "result_based": result_based_qa_df,
    "scorer_list": scorer_list_qa_df,
    "player_goal_count": player_goal_count_qa_df,
    "player_goal_minute": player_goal_minute_qa_df
}

final_qa_parts = []

for source_name, source_df in qa_sources.items():
    prepared_source_df = source_df.copy()
    prepared_source_df["source_dataset"] = source_name
    final_qa_parts.append(prepared_source_df)

final_qa_df = pd.concat(
    final_qa_parts,
    ignore_index=True,
    sort=False
)

final_qa_df = final_qa_df[
    preferred_column_order
]

print(
    "Rebuilt final dataset rows:",
    f"{len(final_qa_df):,}"
)

assert len(final_qa_df) == 194_159
assert final_qa_df["question_id"].is_unique
assert final_qa_df["question"].is_unique
assert final_qa_df["answer"].notna().all()

print("Final dataset recombination passed.")

Rebuilt final dataset rows: 194,159
Final dataset recombination passed.


## Cell 95 — Validate the corrected scorer answers

In [ ]:
corrected_scorer_sample = (
    final_qa_df[
        final_qa_df["intent"].eq(
            "match_scorers"
        )
    ][
        [
            "question_id",
            "match_id",
            "question",
            "answer"
        ]
    ]
    .sample(
        n=25,
        random_state=42
    )
)

display(corrected_scorer_sample)

old_repeated_style_count = int(
    final_qa_df.loc[
        final_qa_df["intent"].eq(
            "match_scorers"
        ),
        "answer"
    ].str.contains(
        r"\b(.+?) for .+ and \1 for ",
        regex=True,
        na=False
    ).sum()
)

scorer_answer_validation = pd.DataFrame([
    {
        "check": "Scorer-list questions",
        "value": int(
            final_qa_df["intent"]
            .eq("match_scorers")
            .sum()
        )
    },
    {
        "check": "Missing scorer-list answers",
        "value": int(
            final_qa_df.loc[
                final_qa_df["intent"].eq(
                    "match_scorers"
                ),
                "answer"
            ].isna().sum()
        )
    },
    {
        "check": "Answers using old repeated-name style",
        "value": old_repeated_style_count
    },
    {
        "check": "Duplicate final question IDs",
        "value": int(
            final_qa_df[
                "question_id"
            ].duplicated().sum()
        )
    },
    {
        "check": "Duplicate final question text",
        "value": int(
            final_qa_df[
                "question"
            ].duplicated().sum()
        )
    }
])

display(scorer_answer_validation)

assert len(final_qa_df) == 194_159
assert final_qa_df["question_id"].is_unique
assert final_qa_df["question"].is_unique
assert old_repeated_style_count == 0

print("Corrected scorer-answer validation passed.")

,question_id,match_id,question,answer
105294,SCORERS_06333,MATCH_25167,Who scored in the match between Togo and Cameroon on 2001-01-28?,Samuel Eto'o scored 1 goal for Cameroon and Patrick M'Boma scored 1 goal for Cameroon.
101881,SCORERS_02920,MATCH_13862,Who scored in the match between Iceland and Republic of Ireland on 1983-09-21?,"Gary Waddock scored 1 goal for Republic of Ireland, Michael Robinson scored 1 goal for Republic of Ireland, and Mickey Walsh scored 1 goal for Rep..."
105127,SCORERS_06166,MATCH_24683,Who scored in the match between Venezuela and Bolivia on 2000-06-28?,"Miguel Mea Vitali scored 1 goal for Venezuela, Ruberth Morán scored 1 goal for Venezuela, Jaime Moreno scored 1 goal for Bolivia, Julio César Bald..."
102551,SCORERS_03590,MATCH_16585,Who scored in the match between Yemen and Syria on 1989-03-10?,Nizar Mahrous scored 1 goal for Syria.
108983,SCORERS_10022,MATCH_35615,Who scored in the match between Niger and Morocco on 2012-01-31?,Younès Belhanda scored 1 goal for Morocco.
105312,SCORERS_06351,MATCH_25225,Who scored in the match between Kuwait and Singapore on 2001-02-21?,Khalaf Al-Mutairi scored 1 goal for Kuwait.
113491,SCORERS_14530,MATCH_47344,Who scored in the match between Kuwait and Afghanistan on 2024-06-11?,Eid Al Rashidi scored 1 goal for Kuwait.
100730,SCORERS_01769,MATCH_09187,Who scored in the match between New Zealand and Tahiti on 1973-02-18?,Alan Vest scored 1 goal for New Zealand and Erroll Bennett scored 1 goal for Tahiti.
111296,SCORERS_12335,MATCH_42501,Who scored in the match between Ukraine and Serbia on 2019-06-07?,"Viktor Tsyhankov scored 2 goals for Ukraine, Yevhen Konoplyanka scored 2 goals for Ukraine, and Roman Yaremchuk scored 1 goal for Ukraine."
106896,SCORERS_07935,MATCH_29314,Who scored in the match between United States and Cuba on 2005-07-07?,"Lester Moré scored 1 goal for Cuba, Clint Dempsey scored 1 goal for United States, Landon Donovan scored 2 goals for United States, and DaMarcus B..."


,check,value
0,Scorer-list questions,15508
1,Missing scorer-list answers,0
2,Answers using old repeated-name style,0
3,Duplicate final question IDs,0
4,Duplicate final question text,0


Corrected scorer-answer validation passed.


## Cell 96 — Complete final-dataset validation

In [ ]:
final_validation_rows = [
    {
        "check": "Total rows",
        "value": len(final_qa_df),
        "expected": 194_159
    },
    {
        "check": "Unique question IDs",
        "value": final_qa_df["question_id"].nunique(),
        "expected": 194_159
    },
    {
        "check": "Unique question text",
        "value": final_qa_df["question"].nunique(),
        "expected": 194_159
    },
    {
        "check": "Missing questions",
        "value": int(
            final_qa_df["question"].isna().sum()
        ),
        "expected": 0
    },
    {
        "check": "Missing answers",
        "value": int(
            final_qa_df["answer"].isna().sum()
        ),
        "expected": 0
    },
    {
        "check": "Blank questions",
        "value": int(
            final_qa_df["question"]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        ),
        "expected": 0
    },
    {
        "check": "Blank answers",
        "value": int(
            final_qa_df["answer"]
            .astype("string")
            .str.strip()
            .eq("")
            .sum()
        ),
        "expected": 0
    }
]

final_validation_summary = pd.DataFrame(
    final_validation_rows
)

final_validation_summary["passed"] = (
    final_validation_summary["value"]
    .eq(final_validation_summary["expected"])
)

display(final_validation_summary)

assert final_validation_summary["passed"].all()

print("Complete final-dataset validation passed.")

,check,value,expected,passed
0,Total rows,194159,194159,True
1,Unique question IDs,194159,194159,True
2,Unique question text,194159,194159,True
3,Missing questions,0,0,True
4,Missing answers,0,0,True
5,Blank questions,0,0,True
6,Blank answers,0,0,True


Complete final-dataset validation passed.


## Cell 97 — Confirm final intent distribution

In [ ]:
final_intent_distribution = (
    final_qa_df["intent"]
    .value_counts()
    .rename_axis("intent")
    .reset_index(name="rows")
)

final_intent_distribution["percentage"] = (
    final_intent_distribution["rows"]
    .div(len(final_qa_df))
    .mul(100)
    .round(2)
)

display(final_intent_distribution)

assert set(final_intent_distribution["intent"]) == {
    "match_winner",
    "match_score",
    "match_scorers",
    "player_match_goal_count",
    "player_match_scoring_minutes"
}

assert final_intent_distribution["rows"].sum() == 194_159

print("Final intent-distribution validation passed.")

,intent,rows,percentage
0,match_winner,49481,25.48
1,match_score,49481,25.48
2,player_match_goal_count,39931,20.57
3,player_match_scoring_minutes,39758,20.48
4,match_scorers,15508,7.99


Final intent-distribution validation passed.


## Cell 98 — Export the complete master dataset

In [ ]:
from pathlib import Path

export_directory = Path(
    "/content/drive/MyDrive/NLP/exports"
)

export_directory.mkdir(
    parents=True,
    exist_ok=True
)

master_csv_path = (
    export_directory
    / "football_qa_master.csv"
)

master_jsonl_path = (
    export_directory
    / "football_qa_master.jsonl"
)

final_qa_df.to_csv(
    master_csv_path,
    index=False,
    encoding="utf-8"
)

final_qa_df.to_json(
    master_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Master CSV:", master_csv_path)
print("Master JSONL:", master_jsonl_path)
print("Exported rows:", f"{len(final_qa_df):,}")

Master CSV: /content/drive/MyDrive/NLP/exports/football_qa_master.csv
Master JSONL: /content/drive/MyDrive/NLP/exports/football_qa_master.jsonl
Exported rows: 194,159


## Cell 99 — Verify the exported files

In [ ]:
exported_csv_df = pd.read_csv(
    master_csv_path,
    low_memory=False
)

exported_jsonl_df = pd.read_json(
    master_jsonl_path,
    lines=True
)

export_validation = pd.DataFrame([
    {
        "file": "CSV",
        "rows": len(exported_csv_df),
        "columns": len(exported_csv_df.columns),
        "question_ids_unique": (
            exported_csv_df["question_id"].is_unique
        ),
        "missing_questions": int(
            exported_csv_df["question"].isna().sum()
        ),
        "missing_answers": int(
            exported_csv_df["answer"].isna().sum()
        )
    },
    {
        "file": "JSONL",
        "rows": len(exported_jsonl_df),
        "columns": len(exported_jsonl_df.columns),
        "question_ids_unique": (
            exported_jsonl_df["question_id"].is_unique
        ),
        "missing_questions": int(
            exported_jsonl_df["question"].isna().sum()
        ),
        "missing_answers": int(
            exported_jsonl_df["answer"].isna().sum()
        )
    }
])

display(export_validation)

assert len(exported_csv_df) == 194_159
assert len(exported_jsonl_df) == 194_159

assert list(exported_csv_df.columns) == list(
    final_qa_df.columns
)

assert list(exported_jsonl_df.columns) == list(
    final_qa_df.columns
)

assert exported_csv_df["question_id"].is_unique
assert exported_jsonl_df["question_id"].is_unique

assert exported_csv_df["question"].notna().all()
assert exported_csv_df["answer"].notna().all()

assert exported_jsonl_df["question"].notna().all()
assert exported_jsonl_df["answer"].notna().all()

print("CSV and JSONL export validation passed.")

,file,rows,columns,question_ids_unique,missing_questions,missing_answers
0,CSV,194159,17,True,0,0
1,JSONL,194159,17,True,0,0


CSV and JSONL export validation passed.


## Cell 100 — Create match-level train, validation, and test splits

In [ ]:
from sklearn.model_selection import GroupShuffleSplit


# First split: 80% training, 20% temporary.
train_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.80,
    random_state=42
)

train_indices, temporary_indices = next(
    train_splitter.split(
        final_qa_df,
        groups=final_qa_df["match_id"]
    )
)

train_df = (
    final_qa_df
    .iloc[train_indices]
    .copy()
    .reset_index(drop=True)
)

temporary_df = (
    final_qa_df
    .iloc[temporary_indices]
    .copy()
    .reset_index(drop=True)
)


# Second split: divide the remaining 20% equally
# into 10% validation and 10% test.
validation_test_splitter = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=42
)

validation_indices, test_indices = next(
    validation_test_splitter.split(
        temporary_df,
        groups=temporary_df["match_id"]
    )
)

validation_df = (
    temporary_df
    .iloc[validation_indices]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    temporary_df
    .iloc[test_indices]
    .copy()
    .reset_index(drop=True)
)

print("Training rows:", f"{len(train_df):,}")
print("Validation rows:", f"{len(validation_df):,}")
print("Test rows:", f"{len(test_df):,}")

print()
print(
    "Training matches:",
    f"{train_df['match_id'].nunique():,}"
)
print(
    "Validation matches:",
    f"{validation_df['match_id'].nunique():,}"
)
print(
    "Test matches:",
    f"{test_df['match_id'].nunique():,}"
)

Training rows: 155,578
Validation rows: 19,223
Test rows: 19,358

Training matches: 39,584
Validation matches: 4,948
Test matches: 4,949


## Cell 101 — Verify that no match leakage exists

In [ ]:
train_match_ids = set(
    train_df["match_id"].unique()
)

validation_match_ids = set(
    validation_df["match_id"].unique()
)

test_match_ids = set(
    test_df["match_id"].unique()
)


train_validation_overlap = (
    train_match_ids & validation_match_ids
)

train_test_overlap = (
    train_match_ids & test_match_ids
)

validation_test_overlap = (
    validation_match_ids & test_match_ids
)


split_leakage_validation = pd.DataFrame([
    {
        "comparison": "Train vs validation",
        "overlapping_matches": len(
            train_validation_overlap
        )
    },
    {
        "comparison": "Train vs test",
        "overlapping_matches": len(
            train_test_overlap
        )
    },
    {
        "comparison": "Validation vs test",
        "overlapping_matches": len(
            validation_test_overlap
        )
    }
])

display(split_leakage_validation)

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(final_qa_df)
)

all_split_question_ids = pd.concat(
    [
        train_df["question_id"],
        validation_df["question_id"],
        test_df["question_id"]
    ],
    ignore_index=True
)

assert len(all_split_question_ids) == 194_159
assert all_split_question_ids.is_unique

print("Match-level split leakage validation passed.")

,comparison,overlapping_matches
0,Train vs validation,0
1,Train vs test,0
2,Validation vs test,0


Match-level split leakage validation passed.


## Cell 102 — Review split sizes and percentages

In [ ]:
split_summary = pd.DataFrame([
    {
        "split": "Train",
        "rows": len(train_df),
        "row_percentage": (
            len(train_df) / len(final_qa_df) * 100
        ),
        "unique_matches": (
            train_df["match_id"].nunique()
        )
    },
    {
        "split": "Validation",
        "rows": len(validation_df),
        "row_percentage": (
            len(validation_df) / len(final_qa_df) * 100
        ),
        "unique_matches": (
            validation_df["match_id"].nunique()
        )
    },
    {
        "split": "Test",
        "rows": len(test_df),
        "row_percentage": (
            len(test_df) / len(final_qa_df) * 100
        ),
        "unique_matches": (
            test_df["match_id"].nunique()
        )
    }
])

total_unique_matches = (
    final_qa_df["match_id"].nunique()
)

split_summary["match_percentage"] = (
    split_summary["unique_matches"]
    .div(total_unique_matches)
    .mul(100)
)

split_summary[
    [
        "row_percentage",
        "match_percentage"
    ]
] = split_summary[
    [
        "row_percentage",
        "match_percentage"
    ]
].round(2)

display(split_summary)

assert split_summary["rows"].sum() == 194_159

assert (
    split_summary["unique_matches"].sum()
    == total_unique_matches
)

print("Split-size validation passed.")

,split,rows,row_percentage,unique_matches,match_percentage
0,Train,155578,80.13,39584,80.0
1,Validation,19223,9.90,4948,10.0
2,Test,19358,9.97,4949,10.0


Split-size validation passed.


## Cell 103 — Compare intent distributions across splits

In [ ]:
split_datasets = {
    "Train": train_df,
    "Validation": validation_df,
    "Test": test_df
}

split_intent_parts = []

for split_name, split_df in split_datasets.items():
    intent_counts = (
        split_df["intent"]
        .value_counts()
        .rename_axis("intent")
        .reset_index(name="rows")
    )

    intent_counts["percentage"] = (
        intent_counts["rows"]
        .div(len(split_df))
        .mul(100)
        .round(2)
    )

    intent_counts.insert(
        0,
        "split",
        split_name
    )

    split_intent_parts.append(intent_counts)

split_intent_distribution = pd.concat(
    split_intent_parts,
    ignore_index=True
)

display(
    split_intent_distribution.sort_values(
        ["intent", "split"]
    ).reset_index(drop=True)
)

expected_intents = set(
    final_qa_df["intent"].unique()
)

for split_name, split_df in split_datasets.items():
    assert set(split_df["intent"].unique()) == (
        expected_intents
    )

print("Every split contains all five intents.")

,split,intent,rows,percentage
0,Test,match_score,4949,25.57
1,Train,match_score,39584,25.44
2,Validation,match_score,4948,25.74
3,Test,match_scorers,1540,7.96
4,Train,match_scorers,12423,7.99
5,Validation,match_scorers,1545,8.04
6,Test,match_winner,4949,25.57
7,Train,match_winner,39584,25.44
8,Validation,match_winner,4948,25.74
9,Test,player_match_goal_count,3976,20.54


Every split contains all five intents.


## Cell 104 — Export train, validation, and test splits

In [ ]:
split_export_directory = (
    export_directory / "splits"
)

split_export_directory.mkdir(
    parents=True,
    exist_ok=True
)

split_datasets = {
    "train": train_df,
    "validation": validation_df,
    "test": test_df
}

split_export_paths = {}

for split_name, split_df in split_datasets.items():
    csv_path = (
        split_export_directory
        / f"football_qa_{split_name}.csv"
    )

    jsonl_path = (
        split_export_directory
        / f"football_qa_{split_name}.jsonl"
    )

    split_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8"
    )

    split_df.to_json(
        jsonl_path,
        orient="records",
        lines=True,
        force_ascii=False
    )

    split_export_paths[split_name] = {
        "csv": csv_path,
        "jsonl": jsonl_path
    }

    print(
        f"{split_name.title()} rows:",
        f"{len(split_df):,}"
    )
    print(" CSV:", csv_path)
    print(" JSONL:", jsonl_path)
    print()

Train rows: 155,578
 CSV: /content/drive/MyDrive/NLP/exports/splits/football_qa_train.csv
 JSONL: /content/drive/MyDrive/NLP/exports/splits/football_qa_train.jsonl

Validation rows: 19,223
 CSV: /content/drive/MyDrive/NLP/exports/splits/football_qa_validation.csv
 JSONL: /content/drive/MyDrive/NLP/exports/splits/football_qa_validation.jsonl

Test rows: 19,358
 CSV: /content/drive/MyDrive/NLP/exports/splits/football_qa_test.csv
 JSONL: /content/drive/MyDrive/NLP/exports/splits/football_qa_test.jsonl



## Cell 105 — Save the match-to-split assignment

In [ ]:
split_assignment_parts = []

for split_name, split_df in split_datasets.items():
    assignment_part = pd.DataFrame({
        "match_id": sorted(
            split_df["match_id"].unique()
        )
    })

    assignment_part["split"] = split_name
    split_assignment_parts.append(assignment_part)

match_split_assignments = pd.concat(
    split_assignment_parts,
    ignore_index=True
)

match_split_assignments = (
    match_split_assignments
    .sort_values("match_id")
    .reset_index(drop=True)
)

match_split_path = (
    split_export_directory
    / "football_qa_match_split_assignments.csv"
)

match_split_assignments.to_csv(
    match_split_path,
    index=False,
    encoding="utf-8"
)

display(
    match_split_assignments["split"]
    .value_counts()
    .rename_axis("split")
    .reset_index(name="matches")
)

print("Assignment file:", match_split_path)

assert len(match_split_assignments) == 49_481
assert match_split_assignments["match_id"].is_unique
assert set(match_split_assignments["split"]) == {
    "train",
    "validation",
    "test"
}

print("Match-to-split assignment saved successfully.")

,split,matches
0,train,39584
1,test,4949
2,validation,4948


Assignment file: /content/drive/MyDrive/NLP/exports/splits/football_qa_match_split_assignments.csv
Match-to-split assignment saved successfully.


## Cell 106 — Reload and validate every exported split

In [ ]:
reloaded_split_datasets = {}
split_export_validation_rows = []

for split_name, expected_df in split_datasets.items():
    paths = split_export_paths[split_name]

    reloaded_csv_df = pd.read_csv(
        paths["csv"],
        low_memory=False
    )

    reloaded_jsonl_df = pd.read_json(
        paths["jsonl"],
        lines=True
    )

    assert len(reloaded_csv_df) == len(expected_df)
    assert len(reloaded_jsonl_df) == len(expected_df)

    assert list(reloaded_csv_df.columns) == list(
        final_qa_df.columns
    )

    assert list(reloaded_jsonl_df.columns) == list(
        final_qa_df.columns
    )

    for exported_df in [
        reloaded_csv_df,
        reloaded_jsonl_df
    ]:
        assert exported_df["question_id"].is_unique
        assert exported_df["question"].notna().all()
        assert exported_df["answer"].notna().all()

    assert set(reloaded_csv_df["match_id"]) == set(
        expected_df["match_id"]
    )

    assert set(reloaded_jsonl_df["match_id"]) == set(
        expected_df["match_id"]
    )

    reloaded_split_datasets[split_name] = (
        reloaded_csv_df
    )

    split_export_validation_rows.append({
        "split": split_name.title(),
        "expected_rows": len(expected_df),
        "csv_rows": len(reloaded_csv_df),
        "jsonl_rows": len(reloaded_jsonl_df),
        "columns": len(reloaded_csv_df.columns),
        "unique_question_ids": (
            reloaded_csv_df["question_id"].is_unique
        ),
        "unique_matches": (
            reloaded_csv_df["match_id"].nunique()
        )
    })

split_export_validation = pd.DataFrame(
    split_export_validation_rows
)

display(split_export_validation)

print("All split files reloaded and validated.")

,split,expected_rows,csv_rows,jsonl_rows,columns,unique_question_ids,unique_matches
0,Train,155578,155578,155578,17,True,39584
1,Validation,19223,19223,19223,17,True,4948
2,Test,19358,19358,19358,17,True,4949


All split files reloaded and validated.


## Cell 107 — Final cross-split validation after reloading

In [ ]:
reloaded_train = reloaded_split_datasets["train"]
reloaded_validation = (
    reloaded_split_datasets["validation"]
)
reloaded_test = reloaded_split_datasets["test"]

reloaded_match_sets = {
    "train": set(reloaded_train["match_id"]),
    "validation": set(
        reloaded_validation["match_id"]
    ),
    "test": set(reloaded_test["match_id"])
}

assert reloaded_match_sets["train"].isdisjoint(
    reloaded_match_sets["validation"]
)

assert reloaded_match_sets["train"].isdisjoint(
    reloaded_match_sets["test"]
)

assert reloaded_match_sets["validation"].isdisjoint(
    reloaded_match_sets["test"]
)

reloaded_question_ids = pd.concat(
    [
        reloaded_train["question_id"],
        reloaded_validation["question_id"],
        reloaded_test["question_id"]
    ],
    ignore_index=True
)

assert len(reloaded_question_ids) == 194_159
assert reloaded_question_ids.is_unique

assert set(reloaded_question_ids) == set(
    final_qa_df["question_id"]
)

print(
    "Final exported-split validation passed:"
)
print("- 194,159 QA pairs preserved")
print("- No duplicate question IDs")
print("- No missing questions")
print("- No match leakage")

Final exported-split validation passed:
- 194,159 QA pairs preserved
- No duplicate question IDs
- No missing questions
- No match leakage


## 9. Version 2 Linguistic Variation and Challenge Evaluation

In [29]:
import sys
import importlib.util
from pathlib import Path

pipeline_path = Path(
    "/content/drive/MyDrive/NLP/src/football_qa_pipeline.py"
)

assert pipeline_path.is_file(), (
    f"Pipeline file not found: {pipeline_path}"
)

sys.modules.pop("football_qa_pipeline", None)

spec = importlib.util.spec_from_file_location(
    "football_qa_pipeline",
    pipeline_path
)

football_qa_pipeline = importlib.util.module_from_spec(spec)
sys.modules["football_qa_pipeline"] = football_qa_pipeline
spec.loader.exec_module(football_qa_pipeline)

print("Loaded from:", football_qa_pipeline.__file__)
print(
    "Version 2 available:",
    hasattr(football_qa_pipeline, "run_v2_pipeline")
)

Loaded from: /content/drive/MyDrive/NLP/src/football_qa_pipeline.py
Version 2 available: True


In [30]:
artifacts_v2 = football_qa_pipeline.run_v2_pipeline(
    data_directory="/content/drive/MyDrive/NLP/data",
    export_directory="/content/drive/MyDrive/NLP/exports_v2",
    enforce_reference_counts=True,
    random_state=42
)

print("Version 2 generation completed.")

Version 2 generation completed.


In [32]:
display(artifacts_v2["validation_summary"])

,dataset,rows,unique_matches,templates,seen_templates
0,train,155578,39584,40,True
1,validation,19223,4948,40,True
2,test,19358,4949,40,True
3,challenge_test,19358,4949,20,False


In [33]:
import pandas as pd

standard_master_v2 = pd.concat(
    [
        artifacts_v2["standard_splits"]["train"],
        artifacts_v2["standard_splits"]["validation"],
        artifacts_v2["standard_splits"]["test"]
    ],
    ignore_index=True
)

print("Standard master rows:", f"{len(standard_master_v2):,}")
print("Challenge-test rows:", f"{len(artifacts_v2['challenge_test']):,}")

old_pgcount_count = (
    standard_master_v2["question"]
    .str.contains(
        r"How often did .* get on the scoresheet",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

old_pgminute_count = (
    standard_master_v2["question"]
    .str.contains(
        r"playing for .* or .*",
        case=False,
        regex=True,
        na=False
    )
    .sum()
)

print("Old PGCOUNT wording:", old_pgcount_count)
print("Old PGMINUTE wording:", old_pgminute_count)

assert len(standard_master_v2) == 194_159
assert old_pgcount_count == 0
assert old_pgminute_count == 0

print("Final Version 2 checks passed.")

Standard master rows: 194,159
Challenge-test rows: 19,358
Old PGCOUNT wording: 0
Old PGMINUTE wording: 0
Final Version 2 checks passed.


In [34]:
%cd /content

!git clone --branch disath-dev \
https://github.com/eshan14git/football-qa-nlp.git

%cd /content/football-qa-nlp

!git branch --show-current
!git status

/content
fatal: destination path 'football-qa-nlp' already exists and is not an empty directory.
/content/football-qa-nlp
disath-dev
On branch disath-dev
Your branch is up to date with 'origin/disath-dev'.

nothing to commit, working tree clean


In [35]:
%cd /content/football-qa-nlp

!git switch disath-dev
!git pull origin disath-dev
!git status

/content/football-qa-nlp
Already on 'disath-dev'
Your branch is up to date with 'origin/disath-dev'.
From https://github.com/eshan14git/football-qa-nlp
 * branch            disath-dev -> FETCH_HEAD
Already up to date.
On branch disath-dev
Your branch is up to date with 'origin/disath-dev'.

nothing to commit, working tree clean


In [36]:
from pathlib import Path

nlp_directory = Path("/content/drive/MyDrive/NLP")

print("Notebook candidates:")
for path in nlp_directory.rglob("*.ipynb"):
    if "shared_pipeline" in path.name.lower():
        print("-", path)

print("\nPipeline candidates:")
for path in nlp_directory.rglob("football_qa_pipeline.py"):
    print("-", path)

Notebook candidates:
- /content/drive/MyDrive/NLP/notebook/disath_football_qa_shared_pipeline.ipynb

Pipeline candidates:
- /content/drive/MyDrive/NLP/src/football_qa_pipeline.py
